In [1]:
# ============================================================
# DATA PREPARATION - BAB 4
# ============================================================
# Kode ini mencakup 5 tahap:
# 1. Baca data dari Google Sheets
# 2. Tampilkan struktur data
# 3. Identifikasi format PAIRED
# 4. Identifikasi data anomali (missing value)
# 5. Siapkan data untuk normalisasi
# ============================================================

import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime

print("="*80)
print("DATA PREPARATION - BAB 4")
print("="*80)

# ------------------------------------------------------------------
# TAHAP 1: BACA DATA DARI GOOGLE SHEETS
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 1: BACA DATA DARI GOOGLE SHEETS")
print("="*80)

CREDENTIALS_PATH = r"C:\Users\fardh\Skripsi\Folder Baru\bim-flight-demo\credentials\credentials.json"
GOOGLE_SHEET_KEY = "1njLUn55TWwYjTQKEmmR97bcwgYaLYdF-0j5vK-vUIRw"

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_PATH, scope)
gc = gspread.authorize(creds)
sheet = gc.open_by_key(GOOGLE_SHEET_KEY)

print("Berhasil terhubung ke Google Sheets")

df_arr = pd.DataFrame(sheet.worksheet("RAW_ARRIVAL").get_all_records())
df_dep = pd.DataFrame(sheet.worksheet("RAW_DEPARTURE").get_all_records())

print(f"\nData berhasil dimuat:")
print(f"   - Arrival   : {len(df_arr):,} baris, {len(df_arr.columns)} kolom")
print(f"   - Departure : {len(df_dep):,} baris, {len(df_dep.columns)} kolom")

# ------------------------------------------------------------------
# TAHAP 2: TAMPILKAN STRUKTUR DATA
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 2: STRUKTUR DATA")
print("="*80)

print("\nKOLOM ARRIVAL:")
print(df_arr.columns.tolist())

print("\nKOLOM DEPARTURE:")
print(df_dep.columns.tolist())

print("\nTIPE DATA ARRIVAL:")
print(df_arr.dtypes)

print("\nTIPE DATA DEPARTURE:")
print(df_dep.dtypes)

print("\nSAMPLE ARRIVAL (5 baris):")
print(df_arr.head())

print("\nSAMPLE DEPARTURE (5 baris):")
print(df_dep.head())

# ------------------------------------------------------------------
# TAHAP 3: IDENTIFIKASI FORMAT PAIRED
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 3: IDENTIFIKASI FORMAT PAIRED")
print("="*80)

# 3.1. Cek PAIRED di Arrival
print("\nPAIRED_FLIGHT - ARRIVAL:")
print(f"   Total baris          : {len(df_arr):,}")
print(f"   PAIRED terisi        : {df_arr['Paired_Flight'].notna().sum():,}")
print(f"   PAIRED kosong        : {df_arr['Paired_Flight'].isna().sum():,}")

# 3.2. Cek PAIRED di Departure
print("\nPAIRED_FLIGHT - DEPARTURE:")
print(f"   Total baris          : {len(df_dep):,}")
print(f"   PAIRED terisi        : {df_dep['Paired_Flight'].notna().sum():,}")
print(f"   PAIRED kosong        : {df_dep['Paired_Flight'].isna().sum():,}")

# 3.3. Analisis Format PAIRED
def detect_paired_format(paired):
    if pd.isna(paired) or str(paired).strip() == "":
        return "kosong"
    if " " in str(paired):
        return "dengan_spasi"
    else:
        return "tanpa_spasi"

arr_formats = df_arr[df_arr["Paired_Flight"].notna()]["Paired_Flight"].apply(detect_paired_format)
dep_formats = df_dep[df_dep["Paired_Flight"].notna()]["Paired_Flight"].apply(detect_paired_format)

print("\nANALISIS FORMAT PAIRED:")
print(f"\n   ARRIVAL:")
print(f"      Dengan spasi : {(arr_formats == 'dengan_spasi').sum():,}")
print(f"      Tanpa spasi  : {(arr_formats == 'tanpa_spasi').sum():,}")
print(f"\n   DEPARTURE:")
print(f"      Dengan spasi : {(dep_formats == 'dengan_spasi').sum():,}")
print(f"      Tanpa spasi  : {(dep_formats == 'tanpa_spasi').sum():,}")

# ------------------------------------------------------------------
# TAHAP 4: IDENTIFIKASI DATA ANOMALI (MISSING VALUE)
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 4: IDENTIFIKASI DATA ANOMALI (MISSING VALUE)")
print("="*80)

# 4.1. Fungsi Identifikasi Missing Value
def analyze_missing(df, name):
    print(f"\n{name}:")
    print("-" * 60)
    
    # Kolom kritis
    critical_cols = ['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 
                     'Landing_Time', 'Onblock_Time', 'BlockOff_Time', 'TakeOff_Time', 'Data_Date']
    
    # Filter kolom yang ada di DataFrame
    available_critical = [col for col in critical_cols if col in df.columns]
    
    for col in available_critical:
        null_count = df[col].isna().sum()
        # Cek empty string untuk object type
        if df[col].dtype == 'object':
            empty_count = (df[col].astype(str).str.strip() == '').sum()
            total_missing = null_count + empty_count
        else:
            total_missing = null_count
        
        pct = (total_missing / len(df)) * 100
        status = "Aman" if total_missing == 0 else "Tidak Aman"
        print(f"   {status} {col:20s} : {total_missing:>6,} ({pct:>6.2f}%)")

# 4.2. Analisis Arrival
print("\nMISSING VALUE :")
analyze_missing(df_arr, "ARRIVAL")

# 4.3. Analisis Departure
analyze_missing(df_dep, "DEPARTURE")

# ------------------------------------------------------------------
# TAHAP 5: SIAPKAN DATA UNTUK NORMALISASI
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 5: SIAPKAN DATA UNTUK NORMALISASI")
print("="*80)

# 5.1. Buat salinan data
arr_clean = df_arr.copy()
dep_clean = df_dep.copy()

print("Salinan data berhasil dibuat")

# 5.2. Bersihkan string
print("\nMembersihkan string...")

arr_clean["Flight_Number"] = arr_clean["Flight_Number"].astype(str).str.strip().str.upper()
dep_clean["Flight_Number"] = dep_clean["Flight_Number"].astype(str).str.strip().str.upper()

arr_clean["Paired_Flight"] = arr_clean["Paired_Flight"].astype(str).str.strip().str.upper()
dep_clean["Paired_Flight"] = dep_clean["Paired_Flight"].astype(str).str.strip().str.upper()

arr_clean["Aircraft_Reg"] = arr_clean["Aircraft_Reg"].astype(str).str.strip().str.upper()
dep_clean["Aircraft_Reg"] = dep_clean["Aircraft_Reg"].astype(str).str.strip().str.upper()

print("String cleaning selesai")

# 5.3. Konversi Data_Date ke datetime
print("\nKonversi Data_Date ke datetime...")

arr_clean["Data_Date"] = pd.to_datetime(arr_clean["Data_Date"], errors="coerce")
dep_clean["Data_Date"] = pd.to_datetime(dep_clean["Data_Date"], errors="coerce")

arr_valid = arr_clean["Data_Date"].notna().sum()
dep_valid = dep_clean["Data_Date"].notna().sum()

print(f"   Arrival   : {arr_valid:,} dari {len(arr_clean):,} ({arr_valid/len(arr_clean)*100:.2f}%)")
print(f"   Departure : {dep_valid:,} dari {len(dep_clean):,} ({dep_valid/len(dep_clean)*100:.2f}%)")

# 5.4. Ekstrak informasi dasar
print("\nEkstrak informasi dasar...")

arr_clean["Day_of_Week"] = arr_clean["Data_Date"].dt.day_name()
dep_clean["Day_of_Week"] = dep_clean["Data_Date"].dt.day_name()

arr_clean["Month"] = arr_clean["Data_Date"].dt.month_name()
dep_clean["Month"] = dep_clean["Data_Date"].dt.month_name()

arr_clean["Is_Weekend"] = arr_clean["Day_of_Week"].isin(["Saturday", "Sunday"])
dep_clean["Is_Weekend"] = dep_clean["Day_of_Week"].isin(["Saturday", "Sunday"])

print("Ekstraksi selesai")

# 5.5. Statistik Data Setelah Persiapan
print("\nSTATISTIK DATA SETELAH PERSIAPAN:")

print(f"\n   ARRIVAL:")
print(f"      Total baris          : {len(arr_clean):,}")
print(f"      Periode              : {arr_clean['Data_Date'].min()} s/d {arr_clean['Data_Date'].max()}")
print(f"      Flight_Number unik   : {arr_clean['Flight_Number'].nunique():,}")
print(f"      PAIRED terisi        : {arr_clean['Paired_Flight'].notna().sum():,}")
print(f"      PAIRED kosong        : {arr_clean['Paired_Flight'].isna().sum():,}")
print(f"      REG terisi           : {arr_clean['Aircraft_Reg'].notna().sum():,}")
print(f"      REG kosong           : {arr_clean['Aircraft_Reg'].isna().sum():,}")

print(f"\n   DEPARTURE:")
print(f"      Total baris          : {len(dep_clean):,}")
print(f"      Periode              : {dep_clean['Data_Date'].min()} s/d {dep_clean['Data_Date'].max()}")
print(f"      Flight_Number unik   : {dep_clean['Flight_Number'].nunique():,}")
print(f"      PAIRED terisi        : {dep_clean['Paired_Flight'].notna().sum():,}")
print(f"      PAIRED kosong        : {dep_clean['Paired_Flight'].isna().sum():,}")
print(f"      REG terisi           : {dep_clean['Aircraft_Reg'].notna().sum():,}")
print(f"      REG kosong           : {dep_clean['Aircraft_Reg'].isna().sum():,}")

# 5.6. Simpan hasil
print("\nMenyimpan hasil sementara...")
# arr_clean.to_csv("arr_clean.csv", index=False)
# dep_clean.to_csv("dep_clean.csv", index=False)

print("Data siap untuk tahap normalisasi")

# ------------------------------------------------------------------
# RINGKASAN AKHIR
# ------------------------------------------------------------------

print("\n" + "="*80)
print("RINGKASAN DATA PREPARATION")
print("="*80)

print(f"""
RINGKASAN DATA PREPARATION

│  TAHAP 1: Baca Data
│     Arrival   : {len(df_arr):,} baris
│     Departure : {len(df_dep):,} baris

│  TAHAP 2: Struktur Data
│     Arrival   : {len(df_arr.columns)} kolom
│     Departure : {len(df_dep.columns)} kolom

│  TAHAP 3: Format PAIRED
│     Arrival   : {(arr_formats == 'dengan_spasi').sum():,} dengan spasi
│     Departure : {(dep_formats == 'dengan_spasi').sum():,} dengan spasi

│  TAHAP 4: Missing Value pada Kolom Kritis
│     Arrival   : {len([c for c in ['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'Landing_Time', 'Onblock_Time'] if (df_arr[c].isna().sum() + (df_arr[c].astype(str).str.strip() == '').sum()) > 0])} kolom
│     Departure : {len([c for c in ['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'BlockOff_Time', 'TakeOff_Time'] if (df_dep[c].isna().sum() + (df_dep[c].astype(str).str.strip() == '').sum()) > 0])} kolom

│  TAHAP 5: Persiapan Normalisasi
│     Arrival   : {len(arr_clean):,} baris siap
│     Departure : {len(dep_clean):,} baris siap

""")

print("\nLANJUT KE TAHAP 6: DATA CLEANING")
print("="*80)

DATA PREPARATION - BAB 4

TAHAP 1: BACA DATA DARI GOOGLE SHEETS
Berhasil terhubung ke Google Sheets

Data berhasil dimuat:
   - Arrival   : 3,498 baris, 20 kolom
   - Departure : 3,503 baris, 20 kolom

TAHAP 2: STRUKTUR DATA

KOLOM ARRIVAL:
['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'Aircraft_Type', 'Origin', 'STA', 'Landing_Time', 'Onblock_Time', 'Stand', 'AVB', 'Adult_Pax', 'Child_Pax', 'Infant_Pax', 'Transit', 'Total_Pax', 'Seat', 'Cargo_Kg', 'Baggage_Kg', 'Data_Date', 'Table_Type']

KOLOM DEPARTURE:
['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'Aircraft_Type', 'Destination', 'STD', 'BlockOff_Time', 'TakeOff_Time', 'Stand', 'AVB', 'Adult_Pax', 'Child_Pax', 'Infant_Pax', 'Transit', 'Total_Pax', 'Seat', 'Cargo_Kg', 'Baggage_Kg', 'Data_Date', 'Table_Type']

TIPE DATA ARRIVAL:
Flight_Number     object
Paired_Flight     object
Aircraft_Reg      object
Aircraft_Type     object
Origin            object
STA               object
Landing_Time      object
Onblock_Time      object

In [5]:
# ============================================================
# TAHAP 6: DATA CLEANING - PENGHAPUSAN DUPLIKAT
# ============================================================

print("\n" + "="*80)
print("TAHAP 6: DATA CLEANING - PENGHAPUSAN DUPLIKAT")
print("="*80)

# ------------------------------------------------------------------
# 1. Identifikasi Duplikat yang Perlu Dihapus
# ------------------------------------------------------------------

print("\nIdentifikasi duplikat yang perlu dihapus...")

# Arrival: Cari duplikat dengan Landing_Time dan Onblock_Time kosong
arr_empty = df_arr[
    (df_arr["Landing_Time"].isna() | (df_arr["Landing_Time"].astype(str).str.strip() == "")) &
    (df_arr["Onblock_Time"].isna() | (df_arr["Onblock_Time"].astype(str).str.strip() == ""))
]

print(f"\n  ARRIVAL - Data dengan Landing_Time dan Onblock_Time kosong:")
print(f"    Jumlah: {len(arr_empty)}")

if len(arr_empty) > 0:
    print("\n    Detail:")
    print(arr_empty[['Flight_Number', 'Data_Date', 'Landing_Time', 'Onblock_Time']].to_string(index=False))

# Departure: Cari duplikat dengan BlockOff_Time dan TakeOff_Time kosong
dep_empty = df_dep[
    (df_dep["BlockOff_Time"].isna() | (df_dep["BlockOff_Time"].astype(str).str.strip() == "")) &
    (df_dep["TakeOff_Time"].isna() | (df_dep["TakeOff_Time"].astype(str).str.strip() == ""))
]

print(f"\n  DEPARTURE - Data dengan BlockOff_Time dan TakeOff_Time kosong:")
print(f"    Jumlah: {len(dep_empty)}")

if len(dep_empty) > 0:
    print("\n    Detail:")
    print(dep_empty[['Flight_Number', 'Data_Date', 'BlockOff_Time', 'TakeOff_Time']].to_string(index=False))

# ------------------------------------------------------------------
# 2. Hapus Data dengan Waktu Kosong
# ------------------------------------------------------------------

print("\nMenghapus data dengan waktu kosong...")

# Arrival
arr_before = len(df_arr)
df_arr_clean = df_arr[
    ~((df_arr["Landing_Time"].isna() | (df_arr["Landing_Time"].astype(str).str.strip() == "")) &
      (df_arr["Onblock_Time"].isna() | (df_arr["Onblock_Time"].astype(str).str.strip() == "")))
].copy()

print(f"  ARRIVAL:")
print(f"    Sebelum : {arr_before:,}")
print(f"    Sesudah : {len(df_arr_clean):,}")
print(f"    Dihapus : {arr_before - len(df_arr_clean):,}")

# Departure
dep_before = len(df_dep)
df_dep_clean = df_dep[
    ~((df_dep["BlockOff_Time"].isna() | (df_dep["BlockOff_Time"].astype(str).str.strip() == "")) &
      (df_dep["TakeOff_Time"].isna() | (df_dep["TakeOff_Time"].astype(str).str.strip() == "")))
].copy()

print(f"\n  DEPARTURE:")
print(f"    Sebelum : {dep_before:,}")
print(f"    Sesudah : {len(df_dep_clean):,}")
print(f"    Dihapus : {dep_before - len(df_dep_clean):,}")

# ------------------------------------------------------------------
# 3. Cek Duplikat dengan Waktu Sama (Valid - Dipertahankan)
# ------------------------------------------------------------------

print("\nCek duplikat dengan waktu sama (valid - dipertahankan)...")

# Arrival
arr_dup_time = df_arr_clean[
    df_arr_clean.duplicated(subset=['Data_Date', 'Landing_Time'], keep=False)
]

print(f"\n  ARRIVAL - Duplikat Landing_Time:")
print(f"    Jumlah: {len(arr_dup_time)}")

if len(arr_dup_time) > 0:
    print("\n    Sample (5 pasang):")
    print(arr_dup_time[['Flight_Number', 'Aircraft_Reg', 'Data_Date', 'Landing_Time']].head(10).to_string(index=False))

# Departure
dep_dup_time = df_dep_clean[
    df_dep_clean.duplicated(subset=['Data_Date', 'BlockOff_Time'], keep=False)
]

print(f"\n  DEPARTURE - Duplikat BlockOff_Time:")
print(f"    Jumlah: {len(dep_dup_time)}")

if len(dep_dup_time) > 0:
    print("\n    Sample (5 pasang):")
    print(dep_dup_time[['Flight_Number', 'Aircraft_Reg', 'Data_Date', 'BlockOff_Time']].head(10).to_string(index=False))

# ------------------------------------------------------------------
# 4. Statistik Akhir
# ------------------------------------------------------------------

print("\n" + "="*80)
print("STATISTIK AKHIR SETELAH CLEANING DUPLIKAT")
print("="*80)
print(f"""
HASIL CLEANING DUPLIKAT

│  ARRIVAL:
│    - Data awal          : 3,498 baris
│    - Waktu kosong       : {arr_before - len(df_arr_clean)} baris
│    - Duplikat valid     : {len(arr_dup_time)} baris (dipertahankan)
│    - HASIL AKHIR        : {len(df_arr_clean):,} baris 

│  DEPARTURE:
│    - Data awal          : 3,503 baris
│    - Waktu kosong       : {dep_before - len(df_dep_clean)} baris
│    - Duplikat valid     : {len(dep_dup_time)} baris (dipertahankan)
│    - HASIL AKHIR        : {len(df_dep_clean):,} baris 
""")

print("\nLANJUT KE TAHAP 7: NORMALISASI DATA")


TAHAP 6: DATA CLEANING - PENGHAPUSAN DUPLIKAT

Identifikasi duplikat yang perlu dihapus...

  ARRIVAL - Data dengan Landing_Time dan Onblock_Time kosong:
    Jumlah: 47

    Detail:
Flight_Number  Data_Date Landing_Time Onblock_Time
       QG 954 2026-02-01                          
       QG 954 2026-02-04                          
       QG 954 2026-02-06                          
      SI 7294 2026-02-07                          
       QG 954 2026-02-08                          
       IU 900 2026-02-13                          
       AK 405 2026-02-22                          
       IU 904 2026-03-12                          
       JT 352 2026-03-12                          
      JT 3229 2026-03-15                          
      ID 6852 2026-03-21                          
       IU 906 2026-03-23                          
      GA 1624 2026-03-29                          
       IU 904 2026-04-04                          
      JT 3904 2026-04-05                          
 

In [7]:
# ============================================================
# TAHAP 7: NORMALISASI DATA
# ============================================================
# Tujuan: Menyeragamkan format Flight_Number, Paired_Flight, 
#         dan Aircraft_Reg untuk memudahkan proses pairing.
# ============================================================

print("\n" + "="*80)
print("TAHAP 7: NORMALISASI DATA")
print("="*80)

# ------------------------------------------------------------------
# 1. Pastikan Data yang Digunakan adalah Hasil Cleaning
# ------------------------------------------------------------------

# Gunakan data hasil cleaning dari TAHAP 6
df_arr = df_arr_clean.copy()
df_dep = df_dep_clean.copy()

print(f"Data yang akan dinormalisasi:")
print(f"   Arrival   : {len(df_arr):,} baris")
print(f"   Departure : {len(df_dep):,} baris")

# ------------------------------------------------------------------
# 2. Fungsi Normalisasi
# ------------------------------------------------------------------

def normalize_flight(flight):
    """
    Normalisasi format flight number.
    - Hapus spasi berlebih
    - Ubah ke uppercase
    - Pastikan format "2 HURUF + SPASI + ANGKA"
    """
    if pd.isna(flight) or flight == "":
        return ""
    
    flight = str(flight).strip().upper()
    
    # Jika sudah ada spasi, return (tapi bersihkan spasi berlebih)
    if " " in flight:
        # Pisahkan huruf dan angka
        parts = flight.split()
        if len(parts) >= 2:
            letters = parts[0]
            numbers = " ".join(parts[1:])
            return f"{letters} {numbers}"
        return flight
    
    # Jika tidak ada spasi dan panjang >= 2
    if len(flight) >= 2:
        # Cek apakah 2 karakter pertama adalah huruf
        if flight[:2].isalpha():
            letters = flight[:2]
            numbers = flight[2:]
            return f"{letters} {numbers}"
    
    return flight

def normalize_reg(reg):
    """
    Normalisasi format aircraft registration.
    - Hapus spasi berlebih
    - Ubah ke uppercase
    """
    if pd.isna(reg) or reg == "":
        return ""
    return str(reg).strip().upper()

# ------------------------------------------------------------------
# 3. Terapkan Normalisasi
# ------------------------------------------------------------------

print("\nNormalisasi Flight_Number...")

df_arr["Flight_Number_Norm"] = df_arr["Flight_Number"].apply(normalize_flight)
df_dep["Flight_Number_Norm"] = df_dep["Flight_Number"].apply(normalize_flight)

print("Flight_Number selesai")

print("\nNormalisasi Paired_Flight...")

df_arr["Paired_Flight_Norm"] = df_arr["Paired_Flight"].apply(normalize_flight)
df_dep["Paired_Flight_Norm"] = df_dep["Paired_Flight"].apply(normalize_flight)

print("Paired_Flight selesai")

print("\nNormalisasi Aircraft_Reg...")

df_arr["Aircraft_Reg_Norm"] = df_arr["Aircraft_Reg"].apply(normalize_reg)
df_dep["Aircraft_Reg_Norm"] = df_dep["Aircraft_Reg"].apply(normalize_reg)

print("Aircraft_Reg selesai")

# ------------------------------------------------------------------
# 4. Cek Hasil Normalisasi
# ------------------------------------------------------------------

print("\nCEK HASIL NORMALISASI:")

print("\n   ARRIVAL - Sample (10 baris):")
print("="*80)
display_cols = ['Flight_Number', 'Flight_Number_Norm', 'Paired_Flight', 'Paired_Flight_Norm', 'Aircraft_Reg', 'Aircraft_Reg_Norm']
print(df_arr[display_cols].head(10).to_string(index=False))

print("\n   DEPARTURE - Sample (10 baris):")
print("="*80)
print(df_dep[display_cols].head(10).to_string(index=False))

# ------------------------------------------------------------------
# 5. Statistik Perubahan
# ------------------------------------------------------------------

print("\nSTATISTIK PERUBAHAN NORMALISASI:")

# Flight_Number yang berubah
arr_fn_changed = df_arr[df_arr["Flight_Number"] != df_arr["Flight_Number_Norm"]]
dep_fn_changed = df_dep[df_dep["Flight_Number"] != df_dep["Flight_Number_Norm"]]

print(f"\n   Flight_Number berubah:")
print(f"      Arrival   : {len(arr_fn_changed):,} ({len(arr_fn_changed)/len(df_arr)*100:.2f}%)")
print(f"      Departure : {len(dep_fn_changed):,} ({len(dep_fn_changed)/len(df_dep)*100:.2f}%)")

# Paired_Flight yang berubah
arr_paired_changed = df_arr[df_arr["Paired_Flight"] != df_arr["Paired_Flight_Norm"]]
dep_paired_changed = df_dep[df_dep["Paired_Flight"] != df_dep["Paired_Flight_Norm"]]

print(f"\n   Paired_Flight berubah:")
print(f"      Arrival   : {len(arr_paired_changed):,} ({len(arr_paired_changed)/len(df_arr)*100:.2f}%)")
print(f"      Departure : {len(dep_paired_changed):,} ({len(dep_paired_changed)/len(df_dep)*100:.2f}%)")

# Aircraft_Reg yang berubah
arr_reg_changed = df_arr[df_arr["Aircraft_Reg"] != df_arr["Aircraft_Reg_Norm"]]
dep_reg_changed = df_dep[df_dep["Aircraft_Reg"] != df_dep["Aircraft_Reg_Norm"]]

print(f"\n   Aircraft_Reg berubah:")
print(f"      Arrival   : {len(arr_reg_changed):,} ({len(arr_reg_changed)/len(df_arr)*100:.2f}%)")
print(f"      Departure : {len(dep_reg_changed):,} ({len(dep_reg_changed)/len(df_dep)*100:.2f}%)")

# ------------------------------------------------------------------
# 6. Tampilkan Contoh Perubahan
# ------------------------------------------------------------------

print("\nCONTOH PERUBAHAN NORMALISASI:")

# Contoh perubahan Flight_Number
if len(arr_fn_changed) > 0:
    print("\n   Flight_Number - Arrival (5 contoh):")
    print(arr_fn_changed[['Flight_Number', 'Flight_Number_Norm']].head(5).to_string(index=False))

if len(dep_fn_changed) > 0:
    print("\n   Flight_Number - Departure (5 contoh):")
    print(dep_fn_changed[['Flight_Number', 'Flight_Number_Norm']].head(5).to_string(index=False))

# ------------------------------------------------------------------
# 7. Simpan Hasil Normalisasi
# ------------------------------------------------------------------

print("\nMenyimpan hasil normalisasi...")

# Simpan ke CSV (opsional, untuk backup)
# df_arr.to_csv("arr_normalized.csv", index=False)
# df_dep.to_csv("dep_normalized.csv", index=False)

# ------------------------------------------------------------------
# 8. Ringkasan Akhir
# ------------------------------------------------------------------

print("\n" + "="*80)
print("RINGKASAN NORMALISASI DATA")
print("="*80)

print(f"""
RINGKASAN NORMALISASI                       

│  ARRIVAL:
│    - Total baris          : {len(df_arr):,}
│    - Flight_Number berubah: {len(arr_fn_changed):,} ({len(arr_fn_changed)/len(df_arr)*100:.2f}%)
│    - Paired_Flight berubah: {len(arr_paired_changed):,} ({len(arr_paired_changed)/len(df_arr)*100:.2f}%)
│    - Aircraft_Reg berubah : {len(arr_reg_changed):,} ({len(arr_reg_changed)/len(df_arr)*100:.2f}%)

│  DEPARTURE:
│    - Total baris          : {len(df_dep):,}
│    - Flight_Number berubah: {len(dep_fn_changed):,} ({len(dep_fn_changed)/len(df_dep)*100:.2f}%)
│    - Paired_Flight berubah: {len(dep_paired_changed):,} ({len(dep_paired_changed)/len(df_dep)*100:.2f}%)
│    - Aircraft_Reg berubah : {len(dep_reg_changed):,} ({len(dep_reg_changed)/len(df_dep)*100:.2f}%)

""")

print("\nNORMALISASI DATA SELESAI!")
print("\nANJUT KE TAHAP 8: FLIGHT PAIR MATCHING")

# ------------------------------------------------------------------
# 9. Persiapan untuk Pairing (Pilih Kolom yang Dibutuhkan)
# ------------------------------------------------------------------

print("\n" + "="*80)
print("PERSIAPAN UNTUK FLIGHT PAIR MATCHING")
print("="*80)

# Pilih kolom yang dibutuhkan untuk pairing
arr_pairing = df_arr[['Flight_Number_Norm', 'Paired_Flight_Norm', 'Aircraft_Reg_Norm', 
                      'Aircraft_Type', 'Origin', 'STA', 'Landing_Time', 'Onblock_Time',
                      'Data_Date']].copy()

dep_pairing = df_dep[['Flight_Number_Norm', 'Paired_Flight_Norm', 'Aircraft_Reg_Norm',
                       'Aircraft_Type', 'Destination', 'STD', 'BlockOff_Time', 'TakeOff_Time',
                       'Data_Date']].copy()

# Rename kolom untuk memudahkan
arr_pairing = arr_pairing.rename(columns={
    'Flight_Number_Norm': 'Flight_Number',
    'Paired_Flight_Norm': 'Paired_Flight',
    'Aircraft_Reg_Norm': 'Aircraft_Reg'
})

dep_pairing = dep_pairing.rename(columns={
    'Flight_Number_Norm': 'Flight_Number',
    'Paired_Flight_Norm': 'Paired_Flight',
    'Aircraft_Reg_Norm': 'Aircraft_Reg'
})

print(f"\nData siap untuk pairing:")
print(f"   Arrival   : {len(arr_pairing):,} baris")
print(f"   Departure : {len(dep_pairing):,} baris")

print("\nKolom Arrival untuk pairing:")
print(arr_pairing.columns.tolist())

print("\nKolom Departure untuk pairing:")
print(dep_pairing.columns.tolist())

print("\nSample Arrival (5 baris):")
print(arr_pairing.head())

print("\nSample Departure (5 baris):")
print(dep_pairing.head())

print("\nData siap untuk TAHAP 8: FLIGHT PAIR MATCHING")
print("="*80)


TAHAP 7: NORMALISASI DATA
Data yang akan dinormalisasi:
   Arrival   : 3,451 baris
   Departure : 3,440 baris

Normalisasi Flight_Number...
Flight_Number selesai

Normalisasi Paired_Flight...
Paired_Flight selesai

Normalisasi Aircraft_Reg...
Aircraft_Reg selesai

CEK HASIL NORMALISASI:

   ARRIVAL - Sample (10 baris):
Flight_Number Flight_Number_Norm Paired_Flight Paired_Flight_Norm Aircraft_Reg Aircraft_Reg_Norm
       QG 952             QG 952        QG 953             QG 953        PKGLG             PKGLG
       IP 350             IP 350        IP 351             IP 351        PKPWQ             PKPWQ
      SI 7306            SI 7306       SI 7200            SI 7200        PKBVR             PKBVR
       IU 812             IU 812        IU 813             IU 813        PKSTU             PKSTU
     SI BVG01           SI BVG01                                         PKBVG             PKBVG
       IU 937             IU 937        IU 862             IU 862        PKSJM             PKSJM

In [9]:
# ============================================================
# TAHAP 8: PERSIAPAN DATA UNTUK FLIGHT PAIR MATCHING
# ============================================================
# Tujuan: Menyiapkan data ARRIVAL dan DEPARTURE dari hasil
#         TAHAP 7 agar siap untuk proses pairing.
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("\n" + "="*80)
print("TAHAP 8: PERSIAPAN DATA UNTUK FLIGHT PAIR MATCHING")
print("="*80)

# ------------------------------------------------------------------
# 1. AMBIL DATA DARI HASIL TAHAP 7
# ------------------------------------------------------------------

# Data hasil normalisasi dari TAHAP 7
# arr_pairing dan dep_pairing sudah tersedia dari TAHAP 7

print(f"\nData dari TAHAP 7:")
print(f"   Arrival   : {len(arr_pairing):,} baris")
print(f"   Departure : {len(dep_pairing):,} baris")

# Buat salinan untuk diproses
arr = arr_pairing.copy()
dep = dep_pairing.copy()

# ------------------------------------------------------------------
# 2. KONVERSI Data_Date KE DATETIME
# ------------------------------------------------------------------

print("\nKonversi Data_Date ke datetime...")

def clean_date(date_val):
    if pd.isna(date_val):
        return np.nan
    date_str = str(date_val).strip()
    date_str = date_str.replace('"', '').replace("'", "").strip()
    if date_str == "":
        return np.nan
    return date_str

# Bersihkan dan konversi
arr["Data_Date_Clean"] = arr["Data_Date"].apply(clean_date)
dep["Data_Date_Clean"] = dep["Data_Date"].apply(clean_date)

arr["Data_Date"] = pd.to_datetime(arr["Data_Date_Clean"], errors="coerce")
dep["Data_Date"] = pd.to_datetime(dep["Data_Date_Clean"], errors="coerce")

# Hapus data dengan tanggal invalid
arr_before = len(arr)
dep_before = len(dep)

arr = arr.dropna(subset=["Data_Date"]).copy()
dep = dep.dropna(subset=["Data_Date"]).copy()

print(f"   Arrival   : {arr_before:,} → {len(arr):,} (dihapus {arr_before - len(arr):,})")
print(f"   Departure : {dep_before:,} → {len(dep):,} (dihapus {dep_before - len(dep):,})")

# Hapus kolom bantu
if "Data_Date_Clean" in arr.columns:
    arr = arr.drop(columns=["Data_Date_Clean"])
if "Data_Date_Clean" in dep.columns:
    dep = dep.drop(columns=["Data_Date_Clean"])

# ------------------------------------------------------------------
# 3. KONVERSI KOLOM WAKTU KE DATETIME.TIME
# ------------------------------------------------------------------

print("\nKonversi kolom waktu ke datetime.time...")

def convert_to_time(time_val):
    if pd.isna(time_val) or str(time_val).strip() == "":
        return np.nan
    
    time_str = str(time_val).strip()
    
    # Jika ada format "1 day, 0:00:00"
    if "day" in time_str.lower():
        try:
            parts = time_str.split(",")
            if len(parts) > 1:
                time_part = parts[1].strip()
                return datetime.strptime(time_part, "%H:%M:%S").time()
        except:
            pass
    
    # Format standar HH:MM:SS
    try:
        return datetime.strptime(time_str, "%H:%M:%S").time()
    except:
        pass
    
    # Format tanpa detik (HH:MM)
    try:
        return datetime.strptime(time_str, "%H:%M").time()
    except:
        pass
    
    return np.nan

# Konversi Arrival
arr["Landing_Time"] = arr["Landing_Time"].apply(convert_to_time)
arr["Onblock_Time"] = arr["Onblock_Time"].apply(convert_to_time)

# Konversi Departure
dep["BlockOff_Time"] = dep["BlockOff_Time"].apply(convert_to_time)
dep["TakeOff_Time"] = dep["TakeOff_Time"].apply(convert_to_time)

# ------------------------------------------------------------------
# 4. HAPUS DATA DENGAN WAKTU INVALID
# ------------------------------------------------------------------

print("\nMenghapus data dengan waktu invalid...")

# Arrival
arr_before = len(arr)
arr = arr.dropna(subset=["Landing_Time", "Onblock_Time"]).copy()
print(f"   Arrival   : {arr_before:,} → {len(arr):,} (dihapus {arr_before - len(arr):,})")

# Departure
dep_before = len(dep)
dep = dep.dropna(subset=["BlockOff_Time", "TakeOff_Time"]).copy()
print(f"   Departure : {dep_before:,} → {len(dep):,} (dihapus {dep_before - len(dep):,})")

# ------------------------------------------------------------------
# 5. VALIDASI WAKTU (LOGIS)
# ------------------------------------------------------------------

print("\nValidasi logika waktu...")

# Arrival: Landing_Time harus <= Onblock_Time
arr_invalid = arr[arr["Landing_Time"] > arr["Onblock_Time"]]
if len(arr_invalid) > 0:
    print(f"   Arrival - Landing > Onblock: {len(arr_invalid):,} kasus")
    # Perbaiki: set Onblock = Landing + 5 menit
    for idx in arr_invalid.index:
        landing = arr.loc[idx, "Landing_Time"]
        if pd.notna(landing):
            new_onblock = (datetime.combine(datetime.today(), landing) + timedelta(minutes=5)).time()
            arr.loc[idx, "Onblock_Time"] = new_onblock
    print(f"      Diperbaiki")

# Departure: TakeOff_Time harus >= BlockOff_Time
dep_invalid = dep[dep["TakeOff_Time"] < dep["BlockOff_Time"]]
if len(dep_invalid) > 0:
    print(f"   Departure - TakeOff < BlockOff: {len(dep_invalid):,} kasus")
    # Perbaiki: set TakeOff = BlockOff + 5 menit
    for idx in dep_invalid.index:
        blockoff = dep.loc[idx, "BlockOff_Time"]
        if pd.notna(blockoff):
            new_takeoff = (datetime.combine(datetime.today(), blockoff) + timedelta(minutes=5)).time()
            dep.loc[idx, "TakeOff_Time"] = new_takeoff
    print(f"     Diperbaiki")

# ------------------------------------------------------------------
# 6. STATISTIK DATA SIAP PAIRING
# ------------------------------------------------------------------

print("\nSTATISTIK DATA SIAP PAIRING:")

print(f"\n   ARRIVAL:")
print(f"      Total baris        : {len(arr):,}")
if len(arr) > 0:
    print(f"      Periode            : {arr['Data_Date'].min()} s/d {arr['Data_Date'].max()}")
print(f"      Flight_Number unik : {arr['Flight_Number'].nunique():,}")
print(f"      PAIRED terisi      : {arr['Paired_Flight'].notna().sum():,}")
print(f"      PAIRED kosong      : {arr['Paired_Flight'].isna().sum():,}")
print(f"      REG terisi         : {arr['Aircraft_Reg'].notna().sum():,}")
print(f"      REG kosong         : {arr['Aircraft_Reg'].isna().sum():,}")

print(f"\n   DEPARTURE:")
print(f"      Total baris        : {len(dep):,}")
if len(dep) > 0:
    print(f"      Periode            : {dep['Data_Date'].min()} s/d {dep['Data_Date'].max()}")
print(f"      Flight_Number unik : {dep['Flight_Number'].nunique():,}")
print(f"      PAIRED terisi      : {dep['Paired_Flight'].notna().sum():,}")
print(f"      PAIRED kosong      : {dep['Paired_Flight'].isna().sum():,}")
print(f"      REG terisi         : {dep['Aircraft_Reg'].notna().sum():,}")
print(f"      REG kosong         : {dep['Aircraft_Reg'].isna().sum():,}")

# ------------------------------------------------------------------
# 7. SIMPAN HASIL TAHAP 8
# ------------------------------------------------------------------

print("\nMenyimpan data siap pairing...")

# Simpan ke variabel untuk TAHAP 9
arr_ready = arr.copy()
dep_ready = dep.copy()

print("\nTAHAP 8 SELESAI!")
print("   Data siap untuk TAHAP 9: FLIGHT PAIR MATCHING")

print("\n" + "="*80)
print("RINGKASAN TAHAP 8:")
print("="*80)
print(f"""

PERSIAPAN DATA UNTUK PAIRING

│  ARRIVAL:
│    - Data dari TAHAP 7    : {len(arr_pairing):,} baris
│    - Tanggal invalid      : {len(arr_pairing) - len(arr) - (arr_pairing['Landing_Time'].isna().sum() if 'Landing_Time' in arr_pairing.columns else 0):,} baris
│    - HASIL AKHIR          : {len(arr):,} baris 

│  DEPARTURE:
│    - Data dari TAHAP 7    : {len(dep_pairing):,} baris
│    - Tanggal invalid      : {len(dep_pairing) - len(dep) - (dep_pairing['BlockOff_Time'].isna().sum() if 'BlockOff_Time' in dep_pairing.columns else 0):,} baris
│    - HASIL AKHIR          : {len(dep):,} baris 

""")

print(" LANJUT KE TAHAP 9: FLIGHT PAIR MATCHING")
print("="*80)


TAHAP 8: PERSIAPAN DATA UNTUK FLIGHT PAIR MATCHING

Data dari TAHAP 7:
   Arrival   : 3,451 baris
   Departure : 3,440 baris

Konversi Data_Date ke datetime...
   Arrival   : 3,451 → 3,451 (dihapus 0)
   Departure : 3,440 → 3,413 (dihapus 27)

Konversi kolom waktu ke datetime.time...

Menghapus data dengan waktu invalid...
   Arrival   : 3,451 → 3,438 (dihapus 13)
   Departure : 3,413 → 3,394 (dihapus 19)

Validasi logika waktu...
   Arrival - Landing > Onblock: 28 kasus
      Diperbaiki
   Departure - TakeOff < BlockOff: 17 kasus
     Diperbaiki

STATISTIK DATA SIAP PAIRING:

   ARRIVAL:
      Total baris        : 3,438
      Periode            : 2026-02-01 00:00:00 s/d 2026-06-30 00:00:00
      Flight_Number unik : 164
      PAIRED terisi      : 3,438
      PAIRED kosong      : 0
      REG terisi         : 3,438
      REG kosong         : 0

   DEPARTURE:
      Total baris        : 3,394
      Periode            : 2026-02-01 00:00:00 s/d 2026-06-30 00:00:00
      Flight_Number unik 

In [11]:
# ============================================================
# TAHAP 9: FLIGHT PAIR MATCHING - FINAL VERSION
# ============================================================
# Tujuan: Memasangkan Arrival dengan Departure berikutnya
#         untuk pesawat yang sama (Aircraft_Reg)
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("\n" + "="*80)
print("TAHAP 9: FLIGHT PAIR MATCHING - FINAL VERSION")
print("="*80)

# ------------------------------------------------------------------
# 1. DATA YANG DIGUNAKAN
# ------------------------------------------------------------------

# Pastikan data yang digunakan adalah hasil TAHAP 8
arr = arr_ready.copy()
dep = dep_ready.copy()

print(f"\nData yang akan dipair:")
print(f"   Arrival   : {len(arr):,} baris")
print(f"   Departure : {len(dep):,} baris")

# ------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------

def safe_str(val):
    if pd.isna(val) or val is None:
        return ""
    return str(val).strip().upper()

def create_pair_key(flight, date):
    """Buat key untuk matching dengan format FLIGHT|YYYY-MM-DD"""
    return f"{safe_str(flight)}|{date.strftime('%Y-%m-%d')}"

def calc_turnaround(onblock_time, blockoff_time):
    """Hitung Turnaround Time dalam menit"""
    if pd.isna(onblock_time) or pd.isna(blockoff_time):
        return np.nan
    if not hasattr(onblock_time, 'hour') or not hasattr(blockoff_time, 'hour'):
        return np.nan
    
    onblock_min = onblock_time.hour * 60 + onblock_time.minute
    blockoff_min = blockoff_time.hour * 60 + blockoff_time.minute
    tat = blockoff_min - onblock_min
    
    # Koreksi negatif (overnight)
    if tat < 0:
        tat += 1440
    
    return tat

def categorize_tat(tat):
    """Kategorikan Turnaround Time"""
    if pd.isna(tat):
        return "Unknown"
    if tat < 30:
        return "Invalid (<30m)"
    elif tat <= 120:
        return "Normal (30-120m)"
    elif tat <= 360:
        return "Medium (2-6h)"
    elif tat <= 720:
        return "Long (6-12h)"
    else:
        return "Very Long (>12h)"

def is_valid_overnight(arrival_date, departure_date, onblock_time, blockoff_time):
    """
    Validasi apakah Arrival → Departure adalah overnight yang valid.
    - Departure harus di hari berikutnya (H+1)
    - Onblock_Time <= 22:00 (malam)
    - BlockOff_Time >= 05:00 (pagi)
    - TAT >= 30 menit
    """
    if pd.isna(onblock_time) or pd.isna(blockoff_time):
        return False
    
    date_diff = (departure_date - arrival_date).days
    
    # Harus beda 1 hari
    if date_diff != 1:
        return False
    
    arr_min = onblock_time.hour * 60 + onblock_time.minute
    dep_min = blockoff_time.hour * 60 + blockoff_time.minute
    
    # Arrival harus <= 22:00 (1320 menit)
    if arr_min > 1320:
        return False
    
    # Departure harus >= 05:00 (300 menit)
    if dep_min < 300:
        return False
    
    # TAT harus >= 30 menit
    tat = dep_min - arr_min + 1440
    return tat >= 30

# ------------------------------------------------------------------
# 3. TAHAP 9.1: SAME-DAY PAIRING
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 9.1: Same-Day Pairing (Tanggal SAMA)")
print("="*80)

# 3.1. Siapkan data
arr_t1 = arr.copy()
dep_t1 = dep.copy()

# 3.2. Hanya ambil yang memiliki Paired_Flight
arr_t1 = arr_t1[arr_t1["Paired_Flight"].notna() & (arr_t1["Paired_Flight"] != "")].copy()
dep_t1 = dep_t1[dep_t1["Paired_Flight"].notna() & (dep_t1["Paired_Flight"] != "")].copy()

print(f"\n Data dengan PAIRED terisi:")
print(f"     Arrival   : {len(arr_t1):,} baris")
print(f"     Departure : {len(dep_t1):,} baris")

# 3.3. Buat key
arr_t1["Pair_Key"] = arr_t1.apply(
    lambda row: create_pair_key(row["Paired_Flight"], row["Data_Date"]), axis=1
)
dep_t1["Pair_Key"] = dep_t1.apply(
    lambda row: create_pair_key(row["Flight_Number"], row["Data_Date"]), axis=1
)

# 3.4. Hapus duplikat key
arr_t1 = arr_t1.drop_duplicates(subset=["Pair_Key"], keep="first")
dep_t1 = dep_t1.drop_duplicates(subset=["Pair_Key"], keep="first")

print(f"\nSetelah hapus duplikat key:")
print(f"     Arrival   : {len(arr_t1):,} baris")
print(f"     Departure : {len(dep_t1):,} baris")

# 3.5. Merge
result_t1 = arr_t1.merge(dep_t1, on="Pair_Key", how="inner", suffixes=("", "_dep"))

print(f"\nHasil merge: {len(result_t1):,} pasangan")

if len(result_t1) > 0:
    # 3.6. Rename kolom
    result_t1 = result_t1.rename(columns={
        "Flight_Number": "Arrival_Flight",
        "Paired_Flight": "Arrival_Paired",
        "Aircraft_Reg": "Arrival_REG",
        "Aircraft_Type": "Arrival_Aircraft",
        "Origin": "Origin",
        "STA": "STA",
        "Landing_Time": "Landing_Time",
        "Onblock_Time": "Onblock_Time",
        "Data_Date": "Arrival_Date",
        "Flight_Number_dep": "Departure_Flight",
        "Paired_Flight_dep": "Departure_Paired",
        "Aircraft_Reg_dep": "Departure_REG",
        "Aircraft_Type_dep": "Departure_Aircraft",
        "Destination": "Destination",
        "STD": "STD",
        "BlockOff_Time": "BlockOff_Time",
        "TakeOff_Time": "TakeOff_Time",
        "Data_Date_dep": "Departure_Date"
    })
    
    # 3.7. Filter: REG harus sama
    before_reg = len(result_t1)
    result_t1 = result_t1[result_t1["Arrival_REG"] == result_t1["Departure_REG"]].copy()
    print(f"REG sama: {len(result_t1):,} pasangan (dihapus {before_reg - len(result_t1):,})")
    
    # 3.8. Hitung TAT
    result_t1["Turnaround_Time"] = result_t1.apply(
        lambda row: calc_turnaround(row["Onblock_Time"], row["BlockOff_Time"]), axis=1
    )
    
    # 3.9. TAT harus >= 30 menit
    before_tat = len(result_t1)
    result_t1 = result_t1[result_t1["Turnaround_Time"] >= 30].copy()
    print(f"TAT >= 30: {len(result_t1):,} pasangan (dihapus {before_tat - len(result_t1):,})")
    
    # 3.10. Tambahkan kategori TAT
    result_t1["TAT_Category"] = result_t1["Turnaround_Time"].apply(categorize_tat)
    result_t1["Is_Very_Long_TAT"] = result_t1["Turnaround_Time"] > 720

print(f"\nTAHAP 9.1: {len(result_t1):,} pasangan")

# ------------------------------------------------------------------
# 4. TAHAP 9.2: OVERNIGHT PAIRING
# ------------------------------------------------------------------

print("\n" + "="*80)
print("TAHAP 9.2: Overnight Pairing (Tanggal BERBEDA)")
print("="*80)

# 4.1. Ambil data yang BELUM terpair di TAHAP 9.1
arr_unpaired = arr[~arr["Flight_Number"].isin(result_t1["Arrival_Flight"])].copy()
dep_unpaired = dep[~dep["Flight_Number"].isin(result_t1["Departure_Flight"])].copy()

print(f"\nData belum terpair:")
print(f"     Arrival   : {len(arr_unpaired):,} baris")
print(f"     Departure : {len(dep_unpaired):,} baris")

# 4.2. Filter: Arrival dengan Onblock_Time <= 22:00 (malam)
def is_evening(time_val):
    if pd.isna(time_val):
        return False
    if not hasattr(time_val, 'hour'):
        return False
    return time_val.hour * 60 + time_val.minute <= 1320  # 22:00

arr_unpaired["Is_Evening"] = arr_unpaired["Onblock_Time"].apply(is_evening)
arr_evening = arr_unpaired[arr_unpaired["Is_Evening"] == True].copy()

print(f"\nArrival malam (<=22:00): {len(arr_evening):,} baris")

# 4.3. Filter: Departure dengan BlockOff_Time >= 05:00 (pagi)
def is_morning(time_val):
    if pd.isna(time_val):
        return False
    if not hasattr(time_val, 'hour'):
        return False
    return time_val.hour * 60 + time_val.minute >= 300  # 05:00

dep_unpaired["Is_Morning"] = dep_unpaired["BlockOff_Time"].apply(is_morning)
dep_morning = dep_unpaired[dep_unpaired["Is_Morning"] == True].copy()

print(f"\n Departure pagi (>=05:00): {len(dep_morning):,} baris")

# 4.4. Buat key dengan mempertimbangkan ±1 hari
arr_evening["Pair_Key"] = arr_evening["Flight_Number"]
dep_morning["Pair_Key"] = dep_morning["Paired_Flight"]

# 4.5. Merge berdasarkan Flight_Key
result_t2_temp = arr_evening.merge(
    dep_morning,
    on="Pair_Key",
    how="inner",
    suffixes=("", "_dep")
)

print(f"\n  Hasil merge: {len(result_t2_temp):,} kandidat")

if len(result_t2_temp) > 0:
    # 4.6. Rename kolom
    result_t2_temp = result_t2_temp.rename(columns={
        "Flight_Number": "Arrival_Flight",
        "Paired_Flight": "Arrival_Paired",
        "Aircraft_Reg": "Arrival_REG",
        "Aircraft_Type": "Arrival_Aircraft",
        "Origin": "Origin",
        "STA": "STA",
        "Landing_Time": "Landing_Time",
        "Onblock_Time": "Onblock_Time",
        "Data_Date": "Arrival_Date",
        "Flight_Number_dep": "Departure_Flight",
        "Paired_Flight_dep": "Departure_Paired",
        "Aircraft_Reg_dep": "Departure_REG",
        "Aircraft_Type_dep": "Departure_Aircraft",
        "Destination": "Destination",
        "STD": "STD",
        "BlockOff_Time": "BlockOff_Time",
        "TakeOff_Time": "TakeOff_Time",
        "Data_Date_dep": "Departure_Date"
    })
    
    # 4.7. Filter: REG harus sama
    before_reg = len(result_t2_temp)
    result_t2_temp = result_t2_temp[result_t2_temp["Arrival_REG"] == result_t2_temp["Departure_REG"]].copy()
    print(f"      REG sama: {len(result_t2_temp):,} kandidat (dihapus {before_reg - len(result_t2_temp):,})")
    
    # 4.8. Filter: Departure harus H+1
    result_t2_temp["Date_Diff"] = (result_t2_temp["Departure_Date"] - result_t2_temp["Arrival_Date"]).dt.days
    before_diff = len(result_t2_temp)
    result_t2_temp = result_t2_temp[result_t2_temp["Date_Diff"] == 1].copy()
    print(f"      H+1: {len(result_t2_temp):,} kandidat (dihapus {before_diff - len(result_t2_temp):,})")
    
    # 4.9. Validasi overnight
    result_t2_temp["Is_Valid_Overnight"] = result_t2_temp.apply(
        lambda row: is_valid_overnight(
            row["Arrival_Date"], row["Departure_Date"],
            row["Onblock_Time"], row["BlockOff_Time"]
        ), axis=1
    )
    before_valid = len(result_t2_temp)
    result_t2_temp = result_t2_temp[result_t2_temp["Is_Valid_Overnight"] == True].copy()
    print(f"      Valid overnight: {len(result_t2_temp):,} kandidat (dihapus {before_valid - len(result_t2_temp):,})")
    
    # 4.10. Hapus duplikat (1 Arrival → 1 Departure)
    before_dup = len(result_t2_temp)
    result_t2_temp = result_t2_temp.drop_duplicates(subset=["Arrival_Flight"], keep="first")
    result_t2_temp = result_t2_temp.drop_duplicates(subset=["Departure_Flight"], keep="first")
    print(f"      Hapus duplikat: {len(result_t2_temp):,} kandidat (dihapus {before_dup - len(result_t2_temp):,})")
    
    # 4.11. Hitung TAT
    result_t2_temp["Turnaround_Time"] = result_t2_temp.apply(
        lambda row: calc_turnaround(row["Onblock_Time"], row["BlockOff_Time"]), axis=1
    )
    
    # 4.12. TAT harus >= 30 menit
    before_tat = len(result_t2_temp)
    result_t2 = result_t2_temp[result_t2_temp["Turnaround_Time"] >= 30].copy()
    print(f"      TAT >= 30: {len(result_t2):,} pasangan (dihapus {before_tat - len(result_t2):,})")
    
    # 4.13. Tambahkan kategori TAT
    if len(result_t2) > 0:
        result_t2["TAT_Category"] = result_t2["Turnaround_Time"].apply(categorize_tat)
        result_t2["Is_Very_Long_TAT"] = result_t2["Turnaround_Time"] > 720

else:
    result_t2 = pd.DataFrame()
    print(f"      Tidak ada kandidat")

print(f"\n   TAHAP 9.2: {len(result_t2):,} pasangan")

# ------------------------------------------------------------------
# 5. GABUNGKAN HASIL
# ------------------------------------------------------------------

all_paired = pd.concat([result_t1, result_t2], ignore_index=True)

print("\n" + "="*80)
print(" HASIL AKHIR FLIGHT PAIR MATCHING")
print("="*80)

print(f"\n   TOTAL PAIRING: {len(all_paired):,} pasangan")
print(f"     - TAHAP 9.1: {len(result_t1):,} pasangan ({len(result_t1)/len(all_paired)*100:.1f}%)")
print(f"     - TAHAP 9.2: {len(result_t2):,} pasangan ({len(result_t2)/len(all_paired)*100:.1f}%)")

# ------------------------------------------------------------------
# 6. STATISTIK
# ------------------------------------------------------------------

if len(all_paired) > 0:
    print(f"\n   STATISTIK PAIRING:")
    print(f"     Arrival unik  : {all_paired['Arrival_Flight'].nunique():,}")
    print(f"     Departure unik: {all_paired['Departure_Flight'].nunique():,}")
    print(f"     TAT Min  : {all_paired['Turnaround_Time'].min():.0f} menit")
    print(f"     TAT Max  : {all_paired['Turnaround_Time'].max():.0f} menit")
    print(f"     TAT Mean : {all_paired['Turnaround_Time'].mean():.0f} menit")
    
    print(f"\n   DISTRIBUSI KATEGORI TAT:")
    tat_dist = all_paired["TAT_Category"].value_counts().sort_index()
    for cat, count in tat_dist.items():
        pct = count / len(all_paired) * 100
        print(f"     {cat:>15} : {count:>5,} ({pct:>5.1f}%)")
    
    print(f"\n   Very Long TAT (>12 jam): {len(all_paired[all_paired['Is_Very_Long_TAT']]):,} pasangan")
    print(f"     (Data ini TETAP DIPERTAHANKAN, hanya ditandai)")

# ------------------------------------------------------------------
# 7. DATA TIDAK TERPAIR
# ------------------------------------------------------------------

arr_unpaired_final = arr[~arr["Flight_Number"].isin(all_paired["Arrival_Flight"])].copy()
dep_unpaired_final = dep[~dep["Flight_Number"].isin(all_paired["Departure_Flight"])].copy()

print(f"\n   DATA TIDAK TERPAIR:")
print(f"     Arrival   : {len(arr_unpaired_final):,} baris ({len(arr_unpaired_final)/len(arr)*100:.1f}%)")
print(f"     Departure : {len(dep_unpaired_final):,} baris ({len(dep_unpaired_final)/len(dep)*100:.1f}%)")

# ------------------------------------------------------------------
# 8. SIMPAN HASIL
# ------------------------------------------------------------------

print("\n" + "="*80)
print(" FLIGHT PAIR MATCHING SELESAI!")
print("="*80)

# Hasil pairing
print(f"\n   Dataset Final:")
print(f"     all_paired : {len(all_paired):,} pasangan")
print(f"     arr_unpaired_final : {len(arr_unpaired_final):,} baris")
print(f"     dep_unpaired_final : {len(dep_unpaired_final):,} baris")


TAHAP 9: FLIGHT PAIR MATCHING - FINAL VERSION

Data yang akan dipair:
   Arrival   : 3,438 baris
   Departure : 3,394 baris

TAHAP 9.1: Same-Day Pairing (Tanggal SAMA)

 Data dengan PAIRED terisi:
     Arrival   : 3,011 baris
     Departure : 2,980 baris

Setelah hapus duplikat key:
     Arrival   : 3,006 baris
     Departure : 2,980 baris

Hasil merge: 2,874 pasangan
REG sama: 2,772 pasangan (dihapus 102)
TAT >= 30: 2,546 pasangan (dihapus 226)

TAHAP 9.1: 2,546 pasangan

TAHAP 9.2: Overnight Pairing (Tanggal BERBEDA)

Data belum terpair:
     Arrival   : 153 baris
     Departure : 237 baris

Arrival malam (<=22:00): 148 baris

 Departure pagi (>=05:00): 229 baris

  Hasil merge: 755 kandidat
      REG sama: 160 kandidat (dihapus 595)
      H+1: 7 kandidat (dihapus 153)
      Valid overnight: 7 kandidat (dihapus 0)
      Hapus duplikat: 4 kandidat (dihapus 3)
      TAT >= 30: 4 pasangan (dihapus 0)

   TAHAP 9.2: 4 pasangan

 HASIL AKHIR FLIGHT PAIR MATCHING

   TOTAL PAIRING: 2,550 

In [13]:
# ============================================================
# INVESTIGASI: REG Sama vs Berbeda di TAHAP 9.2
# ============================================================

# Ambil hasil merge sebelum filter REG
result_t2_temp_before_reg = arr_evening.merge(
    dep_morning,
    on="Pair_Key",
    how="inner",
    suffixes=("", "_dep")
)

print(f"\nINVESTIGASI REG SAMA vs BERBEDA")
print(f"   Total kandidat sebelum filter REG: {len(result_t2_temp_before_reg)}")

# Cek REG sama
result_t2_temp_before_reg["REG_Sama"] = (
    result_t2_temp_before_reg["Aircraft_Reg"] == result_t2_temp_before_reg["Aircraft_Reg_dep"]
)

reg_sama = result_t2_temp_before_reg["REG_Sama"].sum()
reg_berbeda = len(result_t2_temp_before_reg) - reg_sama

print(f"\n  REG Sama    : {reg_sama:,} kandidat")
print(f"    REG Berbeda : {reg_berbeda:,} kandidat (Dihapus)")

# Tampilkan sample REG berbeda
if reg_berbeda > 0:
    print(f"\n    Sample REG Berbeda (10 kandidat):")
    sample = result_t2_temp_before_reg[~result_t2_temp_before_reg["REG_Sama"]].head(10)
    print(sample[["Flight_Number", "Aircraft_Reg", "Flight_Number_dep", "Aircraft_Reg_dep", "Data_Date", "Data_Date_dep"]].to_string(index=False))


INVESTIGASI REG SAMA vs BERBEDA
   Total kandidat sebelum filter REG: 755

  REG Sama    : 160 kandidat
    REG Berbeda : 595 kandidat (Dihapus)

    Sample REG Berbeda (10 kandidat):
Flight_Number Aircraft_Reg Flight_Number_dep Aircraft_Reg_dep  Data_Date Data_Date_dep
      SI 7306        PKBVR           SI 7293            PKBVG 2026-02-01    2026-02-03
      SI 7306        PKBVR           SI 7293            PKVVJ 2026-02-01    2026-02-24
      SI 7306        PKBVR           SI 7293            PKBVG 2026-02-01    2026-03-03
      SI 7306        PKBVR           SI 7293            PKVVO 2026-02-01    2026-03-10
      SI 7306        PKBVR           SI 7293            PKVVM 2026-02-01    2026-03-24
      SI 7306        PKBVR           SI 7293            PKVVO 2026-02-01    2026-04-07
      SI 7306        PKBVR           SI 7293            PKVVO 2026-02-01    2026-04-14
      SI 7306        PKBVR           SI 7293            PKVVO 2026-02-01    2026-04-21
      SI 7306        PKBVR      

In [15]:
# ============================================================
# INVESTIGASI: SELISIH TANGGAL (H+1)
# ============================================================

# Ambil hasil setelah filter REG
result_t2_temp_after_reg = result_t2_temp_before_reg[result_t2_temp_before_reg["REG_Sama"]].copy()

# Hitung selisih tanggal
result_t2_temp_after_reg["Date_Diff"] = (
    result_t2_temp_after_reg["Data_Date_dep"] - result_t2_temp_after_reg["Data_Date"]
).dt.days

print(f"\n INVESTIGASI SELISIH TANGGAL")
print(f"   Total kandidat setelah filter REG: {len(result_t2_temp_after_reg)}")

# Distribusi selisih tanggal
date_diff_dist = result_t2_temp_after_reg["Date_Diff"].value_counts().sort_index()
print(f"\n    Distribusi selisih tanggal:")
for diff, count in date_diff_dist.items():
    print(f"      H{diff:2d} : {count:>5,} kandidat")

# H+1 yang valid
h_plus_1 = result_t2_temp_after_reg[result_t2_temp_after_reg["Date_Diff"] == 1]
print(f"\n    H+1 (valid) : {len(h_plus_1):,} kandidat")
print(f"    Bukan H+1  : {len(result_t2_temp_after_reg) - len(h_plus_1):,} kandidat (Dihapus)")

# Tampilkan sample yang bukan H+1
if len(result_t2_temp_after_reg) - len(h_plus_1) > 0:
    print(f"\n    Sample Bukan H+1 (10 kandidat):")
    sample = result_t2_temp_after_reg[result_t2_temp_after_reg["Date_Diff"] != 1].head(10)
    print(sample[["Flight_Number", "Data_Date", "Flight_Number_dep", "Data_Date_dep", "Date_Diff"]].to_string(index=False))


 INVESTIGASI SELISIH TANGGAL
   Total kandidat setelah filter REG: 160

    Distribusi selisih tanggal:
      H-138 :     1 kandidat
      H-126 :     1 kandidat
      H-123 :     1 kandidat
      H-113 :     1 kandidat
      H-109 :     1 kandidat
      H-98 :     3 kandidat
      H-96 :     1 kandidat
      H-95 :     4 kandidat
      H-87 :     1 kandidat
      H-81 :     2 kandidat
      H-70 :     3 kandidat
      H-68 :     1 kandidat
      H-67 :     2 kandidat
      H-63 :     1 kandidat
      H-60 :     1 kandidat
      H-55 :     1 kandidat
      H-54 :     1 kandidat
      H-53 :     3 kandidat
      H-47 :     1 kandidat
      H-43 :     1 kandidat
      H-42 :     2 kandidat
      H-35 :     3 kandidat
      H-32 :     3 kandidat
      H-28 :     3 kandidat
      H-25 :     5 kandidat
      H-21 :     1 kandidat
      H-14 :     2 kandidat
      H-12 :     1 kandidat
      H-7 :     2 kandidat
      H-6 :     2 kandidat
      H-4 :     5 kandidat
      H 0 :    24 kandida

In [17]:
# ============================================================
# INVESTIGASI: VALID OVERNIGHT
# ============================================================

# Ambil hasil setelah filter H+1
result_t2_temp_after_hplus1 = result_t2_temp_after_reg[result_t2_temp_after_reg["Date_Diff"] == 1].copy()

print(f"\n INVESTIGASI VALID OVERNIGHT")
print(f"   Total kandidat H+1: {len(result_t2_temp_after_hplus1)}")

# Cek valid overnight
result_t2_temp_after_hplus1["Is_Valid_Overnight"] = result_t2_temp_after_hplus1.apply(
    lambda row: is_valid_overnight(
        row["Data_Date"], row["Data_Date_dep"],
        row["Onblock_Time"], row["BlockOff_Time"]
    ), axis=1
)

valid_overnight = result_t2_temp_after_hplus1["Is_Valid_Overnight"].sum()
invalid_overnight = len(result_t2_temp_after_hplus1) - valid_overnight

print(f"\n    Valid Overnight : {valid_overnight:,} kandidat")
print(f"    Invalid        : {invalid_overnight:,} kandidat (Dihapus)")

# Tampilkan detail valid vs invalid
if invalid_overnight > 0:
    print(f"\n    Detail Invalid Overnight:")
    invalid_sample = result_t2_temp_after_hplus1[~result_t2_temp_after_hplus1["Is_Valid_Overnight"]]
    for idx, row in invalid_sample.iterrows():
        arr_min = row["Onblock_Time"].hour * 60 + row["Onblock_Time"].minute if hasattr(row["Onblock_Time"], 'hour') else 0
        dep_min = row["BlockOff_Time"].hour * 60 + row["BlockOff_Time"].minute if hasattr(row["BlockOff_Time"], 'hour') else 0
        print(f"      {row['Flight_Number']} → {row['Flight_Number_dep']} | Arr: {row['Onblock_Time']} | Dep: {row['BlockOff_Time']}")
        print(f"        Arr_min: {arr_min} (<=1320? {arr_min <= 1320}) | Dep_min: {dep_min} (>=300? {dep_min >= 300})")


 INVESTIGASI VALID OVERNIGHT
   Total kandidat H+1: 7

    Valid Overnight : 7 kandidat
    Invalid        : 0 kandidat (Dihapus)


In [19]:
# ============================================================
# INVESTIGASI: DUPLIKAT
# ============================================================

# Ambil hasil setelah filter valid overnight
result_t2_temp_after_valid = result_t2_temp_after_hplus1[result_t2_temp_after_hplus1["Is_Valid_Overnight"]].copy()

print(f"\n INVESTIGASI DUPLIKAT")
print(f"   Total kandidat valid overnight: {len(result_t2_temp_after_valid)}")

# Cek duplikat Arrival
arr_dup = result_t2_temp_after_valid[result_t2_temp_after_valid["Flight_Number"].duplicated(keep=False)]
dep_dup = result_t2_temp_after_valid[result_t2_temp_after_valid["Flight_Number_dep"].duplicated(keep=False)]

print(f"\n    Duplikat Arrival   : {len(arr_dup)} baris (dihapus untuk menjaga 1:1)")
print(f"    Duplikat Departure: {len(dep_dup)} baris (dihapus untuk menjaga 1:1)")

# Tampilkan duplikat
if len(arr_dup) > 0:
    print(f"\n    Sample Duplikat Arrival:")
    print(arr_dup[["Flight_Number", "Data_Date", "Flight_Number_dep", "Data_Date_dep"]].to_string(index=False))


 INVESTIGASI DUPLIKAT
   Total kandidat valid overnight: 7

    Duplikat Arrival   : 5 baris (dihapus untuk menjaga 1:1)
    Duplikat Departure: 5 baris (dihapus untuk menjaga 1:1)

    Sample Duplikat Arrival:
Flight_Number  Data_Date Flight_Number_dep Data_Date_dep
       GA 164 2026-02-05            GA 167    2026-02-06
       SI 201 2026-02-13           SI 7305    2026-02-14
       SI 201 2026-02-20           SI 7305    2026-02-21
       GA 164 2026-03-16            GA 167    2026-03-17
       SI 201 2026-04-17           SI 7305    2026-04-18


In [21]:
# ============================================================
# INVESTIGASI LENGKAP: 24 KANDIDAT H+0
# ============================================================

print("\n" + "="*80)
print("INVESTIGASI LENGKAP: 24 KANDIDAT H+0")
print("="*80)

# ------------------------------------------------------------------
# 1. AMBIL DATA DARI TAHAP 9.2 (SEBELUM FILTER)
# ------------------------------------------------------------------

# Data yang sudah siap dari TAHAP 9.2
arr_evening = arr_unpaired[arr_unpaired["Is_Evening"] == True].copy()
dep_morning = dep_unpaired[dep_unpaired["Is_Morning"] == True].copy()

# Buat key
arr_evening["Pair_Key"] = arr_evening["Flight_Number"]
dep_morning["Pair_Key"] = dep_morning["Paired_Flight"]

# Merge awal (755 kandidat)
result_t2_temp_before_reg = arr_evening.merge(
    dep_morning,
    on="Pair_Key",
    how="inner",
    suffixes=("", "_dep")
)

print(f"\n Total kandidat awal (merge): {len(result_t2_temp_before_reg):,}")

# ------------------------------------------------------------------
# 2. FILTER: REG SAMA
# ------------------------------------------------------------------

result_t2_temp_before_reg["REG_Sama"] = (
    result_t2_temp_before_reg["Aircraft_Reg"] == result_t2_temp_before_reg["Aircraft_Reg_dep"]
)

result_t2_reg_sama = result_t2_temp_before_reg[result_t2_temp_before_reg["REG_Sama"]].copy()

print(f"\n Setelah filter REG sama: {len(result_t2_reg_sama):,} kandidat")
print(f"   (dihapus {len(result_t2_temp_before_reg) - len(result_t2_reg_sama):,} karena REG berbeda)")

# ------------------------------------------------------------------
# 3. HITUNG SELISIH TANGGAL
# ------------------------------------------------------------------

result_t2_reg_sama["Date_Diff"] = (
    result_t2_reg_sama["Data_Date_dep"] - result_t2_reg_sama["Data_Date"]
).dt.days

# ------------------------------------------------------------------
# 4. AMBIL 24 KANDIDAT H+0 (SELISIH 0 HARI)
# ------------------------------------------------------------------

result_t2_h0 = result_t2_reg_sama[result_t2_reg_sama["Date_Diff"] == 0].copy()

print(f"\n Total kandidat H+0 (selisih 0 hari): {len(result_t2_h0)}")

# ------------------------------------------------------------------
# 5. TAMPILKAN DETAIL 24 KANDIDAT
# ------------------------------------------------------------------

print(f"\n DETAIL 24 KANDIDAT H+0:")
print("="*120)
print(result_t2_h0[[
    "Flight_Number", "Data_Date", "Onblock_Time", 
    "Flight_Number_dep", "Data_Date_dep", "BlockOff_Time",
    "Aircraft_Reg", "Aircraft_Reg_dep"
]].to_string(index=False))

# ------------------------------------------------------------------
# 6. CEK STATUS DI TAHAP 9.1
# ------------------------------------------------------------------

# Cek apakah Arrival sudah terpair di TAHAP 9.1
result_t2_h0["Arrival_Sudah_Terpair_9.1"] = result_t2_h0["Flight_Number"].isin(result_t1["Arrival_Flight"])
result_t2_h0["Departure_Sudah_Terpair_9.1"] = result_t2_h0["Flight_Number_dep"].isin(result_t1["Departure_Flight"])

print(f"\n STATUS DI TAHAP 9.1:")
print(f"   Arrival sudah terpair   : {result_t2_h0['Arrival_Sudah_Terpair_9.1'].sum()}")
print(f"   Arrival BELUM terpair   : {(~result_t2_h0['Arrival_Sudah_Terpair_9.1']).sum()}")
print(f"   Departure sudah terpair : {result_t2_h0['Departure_Sudah_Terpair_9.1'].sum()}")
print(f"   Departure BELUM terpair : {(~result_t2_h0['Departure_Sudah_Terpair_9.1']).sum()}")

# ------------------------------------------------------------------
# 7. PISAHKAN YANG BELUM TERPAIR
# ------------------------------------------------------------------

result_t2_h0_unpaired = result_t2_h0[~result_t2_h0["Arrival_Sudah_Terpair_9.1"]].copy()
result_t2_h0_paired = result_t2_h0[result_t2_h0["Arrival_Sudah_Terpair_9.1"]].copy()

print(f"\n KANDIDAT ARRIVAL BELUM TERPAIR:")
if len(result_t2_h0_unpaired) > 0:
    print(f"   Jumlah: {len(result_t2_h0_unpaired)}")
    print(result_t2_h0_unpaired[[
        "Flight_Number", "Data_Date", "Onblock_Time", 
        "Flight_Number_dep", "Data_Date_dep", "BlockOff_Time"
    ]].to_string(index=False))
else:
    print("    Semua Arrival sudah terpair di TAHAP 9.1")

# ------------------------------------------------------------------
# 8. TAMPILKAN SAMPLE YANG SUDAH TERPAIR
# ------------------------------------------------------------------

if len(result_t2_h0_paired) > 0:
    print(f"\n Sample Arrival SUDAH TERPAIR di TAHAP 9.1 (10 data):")
    print(result_t2_h0_paired[[
        "Flight_Number", "Data_Date", "Flight_Number_dep"
    ]].head(10).to_string(index=False))
    
    # Tampilkan dengan siapa mereka terpair
    print(f"\n Detail Pairing di TAHAP 9.1 (sample 5):")
    for idx, row in result_t2_h0_paired.head(5).iterrows():
        arr_flight = row["Flight_Number"]
        dep_flight = row["Flight_Number_dep"]
        # Cari di result_t1
        paired = result_t1[result_t1["Arrival_Flight"] == arr_flight]
        if len(paired) > 0:
            print(f"   {arr_flight} → {dep_flight} (kandidat H+0) sudah terpair dengan {paired['Departure_Flight'].values[0]}")

# ------------------------------------------------------------------
# 9. KESIMPULAN
# ------------------------------------------------------------------

print("\n" + "="*80)
print(" KESIMPULAN INVESTIGASI 24 KANDIDAT H+0")
print("="*80)

if len(result_t2_h0_unpaired) == 0:
    print("""
     SEMUA 24 KANDIDAT SUDAH TERPAIR DI TAHAP 9.1!
    
    Artinya:
    1. Tidak ada data yang terlewat dari TAHAP 9.1
    2. 24 kandidat ini sudah memiliki pasangan yang valid
    3. Filter H+1 di TAHAP 9.2 sudah benar
    4. Hasil 4 pasangan overnight adalah valid
    
     TIDAK PERLU MENAMBAHKAN APA PUN!
    """)
else:
    print(f"""
     ADA {len(result_t2_h0_unpaired)} KANDIDAT YANG BELUM TERPAIR!
    
    Kandidat yang belum terpair:
    """)
    print(result_t2_h0_unpaired[["Flight_Number", "Data_Date", "Flight_Number_dep"]].to_string(index=False))
    print("""
     Apakah Anda ingin menambahkan kandidat ini ke TAHAP 9.1?
    """)


INVESTIGASI LENGKAP: 24 KANDIDAT H+0

 Total kandidat awal (merge): 755

 Setelah filter REG sama: 160 kandidat
   (dihapus 595 karena REG berbeda)

 Total kandidat H+0 (selisih 0 hari): 24

 DETAIL 24 KANDIDAT H+0:
Flight_Number  Data_Date Onblock_Time Flight_Number_dep Data_Date_dep BlockOff_Time Aircraft_Reg Aircraft_Reg_dep
      SI 7306 2026-02-03     08:58:00           SI 7293    2026-02-03      09:21:00        PKBVG            PKBVG
      SI 7306 2026-02-24     08:40:00           SI 7293    2026-02-24      09:03:00        PKVVJ            PKVVJ
      SI 7306 2026-03-03     08:58:00           SI 7293    2026-03-03      09:21:00        PKBVG            PKBVG
      SI 7306 2026-03-10     08:33:00           SI 7293    2026-03-10      08:58:00        PKVVO            PKVVO
     SI BVR03 2026-03-15     12:33:00          SI BVR04    2026-03-15      12:55:00        PKBVR            PKBVR
    GAB T7MPI 2026-03-15     13:49:00         GAB T7MPI    2026-03-15      14:18:00        T7MPI   

In [23]:
# ============================================================
# TAMBAHKAN 24 KANDIDAT H+0 (TERMASUK YANG TAT < 30)
# ============================================================

print("\n" + "="*80)
print("TAMBAHKAN 24 KANDIDAT H+0 (SEMUA)")
print("="*80)

# Ambil 24 kandidat yang belum terpair
result_t2_h0_unpaired = result_t2_h0[~result_t2_h0["Arrival_Sudah_Terpair_9.1"]].copy()

print(f"\n Menambahkan {len(result_t2_h0_unpaired)} kandidat ke TAHAP 9.1...")

# ------------------------------------------------------------------
# 1. RENAME KOLOM AGAR SESUAI DENGAN result_t1
# ------------------------------------------------------------------

result_t2_h0_unpaired = result_t2_h0_unpaired.rename(columns={
    "Flight_Number": "Arrival_Flight",
    "Paired_Flight": "Arrival_Paired",
    "Aircraft_Reg": "Arrival_REG",
    "Aircraft_Type": "Arrival_Aircraft",
    "Origin": "Origin",
    "STA": "STA",
    "Landing_Time": "Landing_Time",
    "Onblock_Time": "Onblock_Time",
    "Data_Date": "Arrival_Date",
    "Flight_Number_dep": "Departure_Flight",
    "Paired_Flight_dep": "Departure_Paired",
    "Aircraft_Reg_dep": "Departure_REG",
    "Aircraft_Type_dep": "Departure_Aircraft",
    "Destination": "Destination",
    "STD": "STD",
    "BlockOff_Time": "BlockOff_Time",
    "TakeOff_Time": "TakeOff_Time",
    "Data_Date_dep": "Departure_Date"
})

# ------------------------------------------------------------------
# 2. HITUNG TAT
# ------------------------------------------------------------------

result_t2_h0_unpaired["Turnaround_Time"] = result_t2_h0_unpaired.apply(
    lambda row: calc_turnaround(row["Onblock_Time"], row["BlockOff_Time"]), axis=1
)

# Koreksi TAT negatif
result_t2_h0_unpaired.loc[result_t2_h0_unpaired["Turnaround_Time"] < 0, "Turnaround_Time"] += 1440

# ------------------------------------------------------------------
# 3. TAMBAHKAN KATEGORI DAN FLAG
# ------------------------------------------------------------------

# Kategori TAT
result_t2_h0_unpaired["TAT_Category"] = result_t2_h0_unpaired["Turnaround_Time"].apply(categorize_tat)

# Flag untuk TAT < 30 (tidak valid secara operasional, tapi tetap dipertahankan)
result_t2_h0_unpaired["Is_Invalid_TAT"] = result_t2_h0_unpaired["Turnaround_Time"] < 30
result_t2_h0_unpaired["Is_Very_Long_TAT"] = result_t2_h0_unpaired["Turnaround_Time"] > 720

# ------------------------------------------------------------------
# 4. TAMPILKAN STATISTIK KANDIDAT YANG DITAMBAHKAN
# ------------------------------------------------------------------

print(f"\n STATISTIK 24 KANDIDAT:")
print(f"   Total ditambahkan: {len(result_t2_h0_unpaired)}")
print(f"   TAT < 30 (Invalid): {(result_t2_h0_unpaired['Is_Invalid_TAT']).sum()}")
print(f"   TAT >= 30 (Valid) : {(~result_t2_h0_unpaired['Is_Invalid_TAT']).sum()}")

# Tampilkan detail
print(f"\n DETAIL 24 KANDIDAT:")
print("="*120)
print(result_t2_h0_unpaired[[
    "Arrival_Flight", "Arrival_Date", "Onblock_Time",
    "Departure_Flight", "Departure_Date", "BlockOff_Time",
    "Turnaround_Time", "TAT_Category", "Is_Invalid_TAT"
]].to_string(index=False))

# ------------------------------------------------------------------
# 5. GABUNGKAN DENGAN result_t1
# ------------------------------------------------------------------

result_t1_updated = pd.concat([result_t1, result_t2_h0_unpaired], ignore_index=True)

print(f"\n HASIL UPDATE TAHAP 9.1:")
print(f"   Sebelum : {len(result_t1)} pasangan")
print(f"   Ditambah: {len(result_t2_h0_unpaired)} pasangan")
print(f"   Sesudah : {len(result_t1_updated)} pasangan")

# Update result_t1
result_t1 = result_t1_updated.copy()

# ------------------------------------------------------------------
# 6. UPDATE TOTAL PAIRING
# ------------------------------------------------------------------

all_paired = pd.concat([result_t1, result_t2], ignore_index=True)

print(f"\n TOTAL PAIRING SETELAH UPDATE:")
print(f"   TAHAP 9.1: {len(result_t1)} pasangan")
print(f"   TAHAP 9.2: {len(result_t2)} pasangan")
print(f"   TOTAL    : {len(all_paired)} pasangan")


TAMBAHKAN 24 KANDIDAT H+0 (SEMUA)

 Menambahkan 24 kandidat ke TAHAP 9.1...

 STATISTIK 24 KANDIDAT:
   Total ditambahkan: 24
   TAT < 30 (Invalid): 23
   TAT >= 30 (Valid) : 1

 DETAIL 24 KANDIDAT:
Arrival_Flight Arrival_Date Onblock_Time Departure_Flight Departure_Date BlockOff_Time  Turnaround_Time     TAT_Category  Is_Invalid_TAT
       SI 7306   2026-02-03     08:58:00          SI 7293     2026-02-03      09:21:00               23   Invalid (<30m)            True
       SI 7306   2026-02-24     08:40:00          SI 7293     2026-02-24      09:03:00               23   Invalid (<30m)            True
       SI 7306   2026-03-03     08:58:00          SI 7293     2026-03-03      09:21:00               23   Invalid (<30m)            True
       SI 7306   2026-03-10     08:33:00          SI 7293     2026-03-10      08:58:00               25   Invalid (<30m)            True
      SI BVR03   2026-03-15     12:33:00         SI BVR04     2026-03-15      12:55:00               22   Invalid (

In [25]:

print("\n" + "="*80)
print("PERBAIKI DUPLIKAT")
print("="*80)

# ------------------------------------------------------------------
# 1. DATA SAAT INI
# ------------------------------------------------------------------

print(f"\n📊 Data sebelum perbaikan:")
print(f"   Total pasangan: {len(result_t1)}")
print(f"   Arrival unik   : {result_t1['Arrival_Flight'].nunique()}")
print(f"   Departure unik : {result_t1['Departure_Flight'].nunique()}")

# ------------------------------------------------------------------
# 2. IDENTIFIKASI DUPLIKAT
# ------------------------------------------------------------------

# Arrival yang muncul lebih dari 1 kali
dup_arr = result_t1[result_t1["Arrival_Flight"].duplicated(keep=False)]
dup_dep = result_t1[result_t1["Departure_Flight"].duplicated(keep=False)]

print(f"\n🔍 Duplikat yang teridentifikasi:")
print(f"   Arrival duplikat   : {len(dup_arr)} baris")
print(f"   Departure duplikat : {len(dup_dep)} baris")

# Tampilkan sample duplikat Arrival
if len(dup_arr) > 0:
    print(f"\n📋 Sample Arrival duplikat (10):")
    sample_dup = dup_arr.groupby("Arrival_Flight").head(2)
    print(sample_dup[["Arrival_Flight", "Arrival_Date", "Departure_Flight", "Turnaround_Time", "TAT_Category"]].head(10).to_string(index=False))


PERBAIKI DUPLIKAT

📊 Data sebelum perbaikan:
   Total pasangan: 2570
   Arrival unik   : 114
   Departure unik : 116

🔍 Duplikat yang teridentifikasi:
   Arrival duplikat   : 2513 baris
   Departure duplikat : 2512 baris

📋 Sample Arrival duplikat (10):
Arrival_Flight Arrival_Date Departure_Flight  Turnaround_Time     TAT_Category
        QG 952   2026-02-01           QG 953               39 Normal (30-120m)
        IP 350   2026-02-01           IP 351               51 Normal (30-120m)
        IU 812   2026-02-01           IU 813               50 Normal (30-120m)
        IU 937   2026-02-01           IU 862               41 Normal (30-120m)
        GA 148   2026-02-01           GA 149               47 Normal (30-120m)
        IP 354   2026-02-01           IP 355               38 Normal (30-120m)
       ID 7109   2026-02-01          ID 7108               51 Normal (30-120m)
        IU 810   2026-02-01           IU 811               48 Normal (30-120m)
        JT 229   2026-02-01       

In [27]:
# ============================================================
# PERBAIKI DUPLIKAT
# ============================================================

print("\n" + "="*80)
print("PERBAIKI DUPLIKAT - (DENGAN TANGGAL)")
print("="*80)

# ------------------------------------------------------------------
# 1. BUAT KEY UNIK: FLIGHT + TANGGAL
# ------------------------------------------------------------------

# Buat key unik untuk Arrival
result_t1["Arrival_Key"] = result_t1["Arrival_Flight"] + "|" + result_t1["Arrival_Date"].dt.strftime("%Y-%m-%d")
result_t1["Departure_Key"] = result_t1["Departure_Flight"] + "|" + result_t1["Departure_Date"].dt.strftime("%Y-%m-%d")

print(f"\n📊 Data sebelum perbaikan:")
print(f"   Total pasangan: {len(result_t1)}")
print(f"   Arrival Key unik   : {result_t1['Arrival_Key'].nunique()}")
print(f"   Departure Key unik : {result_t1['Departure_Key'].nunique()}")

# ------------------------------------------------------------------
# 2. IDENTIFIKASI DUPLIKAT BERDASARKAN KEY
# ------------------------------------------------------------------

# Duplikat Arrival (berdasarkan Arrival_Key)
dup_arr = result_t1[result_t1["Arrival_Key"].duplicated(keep=False)]
dup_dep = result_t1[result_t1["Departure_Key"].duplicated(keep=False)]

print(f"\n🔍 Duplikat yang teridentifikasi:")
print(f"   Arrival duplikat   : {len(dup_arr)} baris")
print(f"   Departure duplikat : {len(dup_dep)} baris")

if len(dup_arr) > 0:
    print(f"\n📋 Sample Arrival duplikat (10):")
    sample_dup = dup_arr.groupby("Arrival_Key").head(2)
    print(sample_dup[["Arrival_Flight", "Arrival_Date", "Departure_Flight", "Departure_Date", "Turnaround_Time"]].head(10).to_string(index=False))


PERBAIKI DUPLIKAT - (DENGAN TANGGAL)

📊 Data sebelum perbaikan:
   Total pasangan: 2570
   Arrival Key unik   : 2570
   Departure Key unik : 2570

🔍 Duplikat yang teridentifikasi:
   Arrival duplikat   : 0 baris
   Departure duplikat : 0 baris


In [29]:
# ============================================================
# DATASET FINAL TAHAP 9
# ============================================================

print("\n" + "="*80)
print("DATASET FINAL TAHAP 9")
print("="*80)

print(f"\n📊 TOTAL PAIRING: {len(all_paired):,} pasangan")

print(f"\n📋 KOLOM YANG TERSEDIA:")
print(all_paired.columns.tolist())

print(f"\n📄 SAMPLE DATA (5 baris):")
print(all_paired[["Arrival_Flight", "Arrival_Date", "Departure_Flight", "Departure_Date", "Turnaround_Time", "TAT_Category"]].head())

print(f"\n📊 STATISTIK:")
print(f"   Arrival unik  : {all_paired['Arrival_Flight'].nunique()}")
print(f"   Departure unik: {all_paired['Departure_Flight'].nunique()}")
print(f"   TAT Min  : {all_paired['Turnaround_Time'].min():.0f} menit")
print(f"   TAT Max  : {all_paired['Turnaround_Time'].max():.0f} menit")
print(f"   TAT Mean : {all_paired['Turnaround_Time'].mean():.0f} menit")


DATASET FINAL TAHAP 9

📊 TOTAL PAIRING: 2,574 pasangan

📋 KOLOM YANG TERSEDIA:
['Arrival_Flight', 'Arrival_Paired', 'Arrival_REG', 'Arrival_Aircraft', 'Origin', 'STA', 'Landing_Time', 'Onblock_Time', 'Arrival_Date', 'Pair_Key', 'Departure_Flight', 'Departure_Paired', 'Departure_REG', 'Departure_Aircraft', 'Destination', 'STD', 'BlockOff_Time', 'TakeOff_Time', 'Departure_Date', 'Turnaround_Time', 'TAT_Category', 'Is_Very_Long_TAT', 'Is_Evening', 'Is_Morning', 'REG_Sama', 'Date_Diff', 'Arrival_Sudah_Terpair_9.1', 'Departure_Sudah_Terpair_9.1', 'Is_Invalid_TAT', 'Is_Valid_Overnight']

📄 SAMPLE DATA (5 baris):
  Arrival_Flight Arrival_Date Departure_Flight Departure_Date  \
0         QG 952   2026-02-01           QG 953     2026-02-01   
1         IP 350   2026-02-01           IP 351     2026-02-01   
2         IU 812   2026-02-01           IU 813     2026-02-01   
3         IU 937   2026-02-01           IU 862     2026-02-01   
4         GA 148   2026-02-01           GA 149     2026-02-0

In [31]:
# ============================================================
# 1. DATA YANG DIGUNAKAN
# ============================================================

df_fe = all_paired.copy()

print(f"\n📊 Data untuk Feature Engineering:")
print(f"   Total baris : {len(df_fe):,}")
print(f"   Total kolom : {len(df_fe.columns)}")

# Cek kolom yang tersedia
print(f"\n📋 KOLOM YANG TERSEDIA:")
print(df_fe.columns.tolist())

# ============================================================
# 2. CEK KOLOM STAND
# ============================================================

# Cek apakah kolom Stand ada
if 'Stand' not in df_fe.columns:
    print("\n⚠️ Kolom 'Stand' tidak ditemukan di all_paired!")
    print("   Mengambil Stand dari data asli...")
    
    # Ambil Stand dari data departure (asumsi: Stand ada di dep_ready)
    # Cek apakah Stand ada di dep_ready
    if 'Stand' in dep_ready.columns:
        # Gabungkan Stand ke df_fe berdasarkan Departure_Flight dan Departure_Date
        dep_stand = dep_ready[['Flight_Number', 'Data_Date', 'Stand']].copy()
        dep_stand = dep_stand.rename(columns={
            'Flight_Number': 'Departure_Flight',
            'Data_Date': 'Departure_Date'
        })
        
        # Merge ke df_fe
        df_fe = df_fe.merge(
            dep_stand,
            on=['Departure_Flight', 'Departure_Date'],
            how='left'
        )
        print(f"   ✅ Stand berhasil ditambahkan dari dep_ready")
    else:
        print("   ⚠️ Kolom Stand juga tidak ditemukan di dep_ready!")
        # Buat kolom Stand dengan nilai default
        df_fe["Stand"] = "UNKNOWN"
else:
    print(f"\n✅ Kolom Stand tersedia: {df_fe['Stand'].nunique()} stand unik")

# ============================================================
# 3. CEK FORMAT KOLOM WAKTU
# ============================================================

print("\n🔍 CEK FORMAT KOLOM WAKTU:")
print(f"   STA dtype        : {df_fe['STA'].dtype}")
print(f"   Landing_Time dtype: {df_fe['Landing_Time'].dtype}")
print(f"   STD dtype         : {df_fe['STD'].dtype}")
print(f"   BlockOff_Time dtype: {df_fe['BlockOff_Time'].dtype}")
print(f"   TakeOff_Time dtype: {df_fe['TakeOff_Time'].dtype}")

# ============================================================
# 4. FUNGSI-FUNGSI PEMBANTU
# ============================================================

def safe_time_to_minutes(time_val):
    """Konversi waktu ke menit dengan aman."""
    if pd.isna(time_val):
        return np.nan
    if hasattr(time_val, 'hour'):
        return time_val.hour * 60 + time_val.minute
    if isinstance(time_val, str):
        try:
            parts = time_val.split(':')
            if len(parts) >= 2:
                return int(parts[0]) * 60 + int(parts[1])
        except:
            pass
    return np.nan

def get_hour(time_val):
    """Ekstrak jam dari waktu"""
    if pd.isna(time_val):
        return np.nan
    if hasattr(time_val, 'hour'):
        return time_val.hour
    if isinstance(time_val, str):
        try:
            parts = time_val.split(':')
            return int(parts[0])
        except:
            pass
    return np.nan

def calc_delay(actual, scheduled):
    """Hitung delay dalam menit."""
    actual_min = safe_time_to_minutes(actual)
    scheduled_min = safe_time_to_minutes(scheduled)
    if pd.isna(actual_min) or pd.isna(scheduled_min):
        return np.nan
    delay = actual_min - scheduled_min
    if delay > 720:
        delay -= 1440
    if delay < -720:
        delay += 1440
    return delay

def calc_turnaround(onblock_time, blockoff_time):
    """Hitung Turnaround Time dalam menit."""
    onblock_min = safe_time_to_minutes(onblock_time)
    blockoff_min = safe_time_to_minutes(blockoff_time)
    if pd.isna(onblock_min) or pd.isna(blockoff_min):
        return np.nan
    tat = blockoff_min - onblock_min
    if tat < 0:
        tat += 1440
    return tat

def extract_airline(flight_number):
    """Ekstrak 2 huruf pertama dari Flight Number."""
    if pd.isna(flight_number) or flight_number == "":
        return "UNKNOWN"
    flight_str = str(flight_number).strip()
    if len(flight_str) >= 2 and flight_str[:2].isalpha():
        return flight_str[:2].upper()
    return "UNKNOWN"

def is_peak_hour(hour):
    """Peak Hour: 06.00-09.00 atau 16.00-19.00"""
    if pd.isna(hour):
        return False
    return (6 <= hour <= 9) or (16 <= hour <= 19)

def get_departure_period(hour):
    """Kategorisasi jam keberangkatan."""
    if pd.isna(hour):
        return "Unknown"
    if 6 <= hour <= 9:
        return "Morning Peak"
    elif 10 <= hour <= 15:
        return "Midday"
    elif 16 <= hour <= 19:
        return "Evening Peak"
    else:
        return "Night"

def categorize_delay_pm89(delay):
    """Kategori delay berdasarkan PM 89 Tahun 2015."""
    if pd.isna(delay):
        return "Unknown"
    if delay <= 15:
        return "On Time"
    elif delay <= 30:
        return "Minor Delay"
    elif delay <= 60:
        return "Moderate Delay"
    else:
        return "Major Delay"


📊 Data untuk Feature Engineering:
   Total baris : 2,574
   Total kolom : 30

📋 KOLOM YANG TERSEDIA:
['Arrival_Flight', 'Arrival_Paired', 'Arrival_REG', 'Arrival_Aircraft', 'Origin', 'STA', 'Landing_Time', 'Onblock_Time', 'Arrival_Date', 'Pair_Key', 'Departure_Flight', 'Departure_Paired', 'Departure_REG', 'Departure_Aircraft', 'Destination', 'STD', 'BlockOff_Time', 'TakeOff_Time', 'Departure_Date', 'Turnaround_Time', 'TAT_Category', 'Is_Very_Long_TAT', 'Is_Evening', 'Is_Morning', 'REG_Sama', 'Date_Diff', 'Arrival_Sudah_Terpair_9.1', 'Departure_Sudah_Terpair_9.1', 'Is_Invalid_TAT', 'Is_Valid_Overnight']

⚠️ Kolom 'Stand' tidak ditemukan di all_paired!
   Mengambil Stand dari data asli...
   ⚠️ Kolom Stand juga tidak ditemukan di dep_ready!

🔍 CEK FORMAT KOLOM WAKTU:
   STA dtype        : object
   Landing_Time dtype: object
   STD dtype         : object
   BlockOff_Time dtype: object
   TakeOff_Time dtype: object


In [33]:
# ============================================================
# CEK KOLOM YANG TERSEDIA DI df_dep_clean
# ============================================================

print("\n" + "="*80)
print("CEK KOLOM DI df_dep_clean")
print("="*80)

print(f"\n📋 Kolom yang tersedia di df_dep_clean:")
print(df_dep_clean.columns.tolist())

# Cek kolom yang mirip dengan Flight_Number
flight_cols = [col for col in df_dep_clean.columns if 'Flight' in col or 'flight' in col]
print(f"\n📋 Kolom yang mengandung 'Flight':")
print(flight_cols)

# Cek kolom Stand
if 'Stand' in df_dep_clean.columns:
    print("\n✅ Kolom 'Stand' tersedia di df_dep_clean")
else:
    print("\n⚠️ Kolom 'Stand' TIDAK tersedia di df_dep_clean")


CEK KOLOM DI df_dep_clean

📋 Kolom yang tersedia di df_dep_clean:
['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'Aircraft_Type', 'Destination', 'STD', 'BlockOff_Time', 'TakeOff_Time', 'Stand', 'AVB', 'Adult_Pax', 'Child_Pax', 'Infant_Pax', 'Transit', 'Total_Pax', 'Seat', 'Cargo_Kg', 'Baggage_Kg', 'Data_Date', 'Table_Type']

📋 Kolom yang mengandung 'Flight':
['Flight_Number', 'Paired_Flight']

✅ Kolom 'Stand' tersedia di df_dep_clean


In [45]:
# ============================================================
# TAHAP 10: FEATURE ENGINEERING - FINAL VERSION (DIPERBAIKI)
# ============================================================

print("\n" + "="*80)
print("TAHAP 10: FEATURE ENGINEERING - FINAL VERSION (DIPERBAIKI)")
print("="*80)

# ============================================================
# 1. DATA YANG DIGUNAKAN
# ============================================================

df_fe = all_paired.copy()

print(f"\n Data untuk Feature Engineering:")
print(f"   Total baris : {len(df_fe):,}")
print(f"   Total kolom : {len(df_fe.columns)}")

# ============================================================
# 2. TAMBAHKAN KOLOM STAND
# ============================================================

print("\n" + "-"*40)
print(" MENAMBAHKAN KOLOM STAND")
print("-"*40)

# Cek apakah kolom Stand sudah ada di df_fe
if 'Stand' in df_fe.columns:
    print("Kolom Stand sudah tersedia di df_fe")
else:
    print("Kolom Stand tidak ditemukan di df_fe, mengambil dari data mentah...")
    
    # Cek kolom di df_dep_clean
    flight_col = None
    date_col = None
    stand_col = None
    
    for col in df_dep_clean.columns:
        if 'Flight' in col and 'Number' in col:
            flight_col = col
        if 'Data_Date' in col:
            date_col = col
        if 'Stand' in col:
            stand_col = col
    
    print(f"\nKolom yang teridentifikasi di df_dep_clean:")
    print(f"   Flight Number : {flight_col}")
    print(f"   Data Date     : {date_col}")
    print(f"   Stand         : {stand_col}")
    
    if flight_col and date_col and stand_col:
        print(f"\nMengambil Stand dari df_dep_clean")
        
        dep_stand = df_dep_clean[[flight_col, date_col, stand_col]].copy()
        dep_stand = dep_stand.rename(columns={
            flight_col: 'Departure_Flight',
            date_col: 'Departure_Date',
            stand_col: 'Stand'
        })
        
        # Konversi ke datetime
        dep_stand['Departure_Date'] = pd.to_datetime(dep_stand['Departure_Date'], errors='coerce')
        dep_stand = dep_stand.dropna(subset=['Departure_Date'])
        
        # Merge
        df_fe = df_fe.merge(
            dep_stand,
            on=['Departure_Flight', 'Departure_Date'],
            how='left'
        )
        
        # Cek apakah kolom Stand ada
        if 'Stand' in df_fe.columns:
            print(f"\n Stand berhasil ditambahkan dari df_dep_clean")
            print(f"   Total Stand unik: {df_fe['Stand'].nunique()}")
        else:
            print("\n Kolom Stand tidak muncul setelah merge!")
            print("   Membuat kolom Stand dengan nilai default 'UNKNOWN'")
            df_fe["Stand"] = "UNKNOWN"
    else:
        print("\nKolom Stand tidak lengkap di df_dep_clean")
        print("   Membuat kolom Stand dengan nilai default 'UNKNOWN'")
        df_fe["Stand"] = "UNKNOWN"

# ============================================================
# 3. CEK KOLOM Stand SETELAH PROSES
# ============================================================

print(f"\n Hasil kolom Stand:")
if 'Stand' in df_fe.columns:
    print(f"   Total baris: {len(df_fe):,}")
    print(f"   Stand unik : {df_fe['Stand'].nunique()}")
    if df_fe['Stand'].nunique() > 1:
        print(df_fe["Stand"].value_counts().head(10))
else:
    print("    Kolom Stand TIDAK ADA! Membuat dengan nilai default.")
    df_fe["Stand"] = "UNKNOWN"

# ============================================================
# 4. CEK FORMAT KOLOM WAKTU
# ============================================================

print("\n CEK FORMAT KOLOM WAKTU:")
time_cols = ['STA', 'Landing_Time', 'STD', 'BlockOff_Time', 'TakeOff_Time']
for col in time_cols:
    if col in df_fe.columns:
        print(f"   {col:15s} dtype : {df_fe[col].dtype}")

# ============================================================
# 5. FUNGSI-FUNGSI PEMBANTU
# ============================================================

def safe_time_to_minutes(time_val):
    if pd.isna(time_val):
        return np.nan
    if hasattr(time_val, 'hour'):
        return time_val.hour * 60 + time_val.minute
    if isinstance(time_val, str):
        try:
            parts = time_val.split(':')
            if len(parts) >= 2:
                return int(parts[0]) * 60 + int(parts[1])
        except:
            pass
    return np.nan

def get_hour(time_val):
    if pd.isna(time_val):
        return np.nan
    if hasattr(time_val, 'hour'):
        return time_val.hour
    if isinstance(time_val, str):
        try:
            parts = time_val.split(':')
            return int(parts[0])
        except:
            pass
    return np.nan

def calc_delay(actual, scheduled):
    actual_min = safe_time_to_minutes(actual)
    scheduled_min = safe_time_to_minutes(scheduled)
    if pd.isna(actual_min) or pd.isna(scheduled_min):
        return np.nan
    delay = actual_min - scheduled_min
    if delay > 720:
        delay -= 1440
    if delay < -720:
        delay += 1440
    return delay

def calc_turnaround(onblock_time, blockoff_time):
    onblock_min = safe_time_to_minutes(onblock_time)
    blockoff_min = safe_time_to_minutes(blockoff_time)
    if pd.isna(onblock_min) or pd.isna(blockoff_min):
        return np.nan
    tat = blockoff_min - onblock_min
    if tat < 0:
        tat += 1440
    return tat

def extract_airline(flight_number):
    if pd.isna(flight_number) or flight_number == "":
        return "UNKNOWN"
    flight_str = str(flight_number).strip()
    if len(flight_str) >= 2 and flight_str[:2].isalpha():
        return flight_str[:2].upper()
    return "UNKNOWN"

def is_peak_hour(hour):
    if pd.isna(hour):
        return False
    return (6 <= hour <= 9) or (16 <= hour <= 19)

def get_departure_period(hour):
    if pd.isna(hour):
        return "Unknown"
    if 6 <= hour <= 9:
        return "Morning Peak"
    elif 10 <= hour <= 15:
        return "Midday"
    elif 16 <= hour <= 19:
        return "Evening Peak"
    else:
        return "Night"

# ============================================================
# FUNGSI DELAY CATEGORY YANG DIPERBAIKI
# ============================================================

def categorize_delay_correct(delay):
    """
    Kategori delay berdasarkan aturan yang benar:
    - OTP          : ≤ 29 menit
    - Kategori 1   : 30 - 60 menit
    - Kategori 2   : 61 - 120 menit
    - Kategori 3   : 121 - 180 menit
    - Kategori 4   : 181 - 240 menit
    - Kategori 5   : > 240 menit
    - Kategori 6   : Pembatalan (tidak termasuk dalam dataset)
    """
    if pd.isna(delay):
        return "Unknown"
    if delay <= 29:
        return "OTP"
    elif delay <= 60:
        return "Kategori 1"
    elif delay <= 120:
        return "Kategori 2"
    elif delay <= 180:
        return "Kategori 3"
    elif delay <= 240:
        return "Kategori 4"
    else:
        return "Kategori 5"

# ============================================================
# 6. FEATURE 1: AIRLINE
# ============================================================

df_fe["Airline"] = df_fe["Departure_Flight"].apply(extract_airline)
print(f"\n 1. Airline: {df_fe['Airline'].nunique()} maskapai")
print(df_fe["Airline"].value_counts().head(8))

# ============================================================
# 7. FEATURE 2: AIRCRAFT TYPE
# ============================================================

df_fe["Aircraft_Type"] = df_fe["Departure_Aircraft"]
print(f"\n 2. Aircraft Type: {df_fe['Aircraft_Type'].nunique()} tipe")
print(df_fe["Aircraft_Type"].value_counts().head(8))

# ============================================================
# 8. FEATURE 3: ARRIVAL DELAY
# ============================================================

df_fe["Arrival_Delay"] = df_fe.apply(
    lambda row: calc_delay(row["Landing_Time"], row["STA"]), axis=1
)

valid_count = df_fe['Arrival_Delay'].notna().sum()
print(f"\n 3. Arrival Delay: {valid_count:,} data valid")
if valid_count > 0:
    print(f"   Min  : {df_fe['Arrival_Delay'].min():.0f} menit")
    print(f"   Max  : {df_fe['Arrival_Delay'].max():.0f} menit")
    print(f"   Mean : {df_fe['Arrival_Delay'].mean():.0f} menit")

# ============================================================
# 9. FEATURE 4: TURNAROUND TIME
# ============================================================

df_fe["Turnaround_Time"] = df_fe.apply(
    lambda row: calc_turnaround(row["Onblock_Time"], row["BlockOff_Time"]), axis=1
)

valid_count = df_fe['Turnaround_Time'].notna().sum()
print(f"\n 4. Turnaround Time: {valid_count:,} data valid")
if valid_count > 0:
    print(f"   Min  : {df_fe['Turnaround_Time'].min():.0f} menit")
    print(f"   Max  : {df_fe['Turnaround_Time'].max():.0f} menit")
    print(f"   Mean : {df_fe['Turnaround_Time'].mean():.0f} menit")

# ============================================================
# 10. FEATURE 5: DEPARTURE HOUR
# ============================================================

df_fe["Departure_Hour"] = df_fe["BlockOff_Time"].apply(get_hour)
print(f"\n 5. Departure Hour: {df_fe['Departure_Hour'].notna().sum():,} data valid")

# ============================================================
# 11. FEATURE 6: DEPARTURE PERIOD
# ============================================================

df_fe["Departure_Period"] = df_fe["Departure_Hour"].apply(get_departure_period)
print(f"\n 6. Departure Period:")
print(df_fe["Departure_Period"].value_counts())

# ============================================================
# 12. FEATURE 7: PEAK HOUR
# ============================================================

df_fe["Peak_Hour"] = df_fe["Departure_Hour"].apply(is_peak_hour)
peak_count = df_fe["Peak_Hour"].sum()
print(f"\n 7. Peak Hour:")
print(f"   TRUE (Peak)   : {peak_count:,} ({peak_count/len(df_fe)*100:.1f}%)")
print(f"   FALSE (Off Peak): {len(df_fe)-peak_count:,} ({(len(df_fe)-peak_count)/len(df_fe)*100:.1f}%)")

# ============================================================
# 13. FEATURE 8: IS WEEKEND
# ============================================================

df_fe["Day_of_Week"] = df_fe["Departure_Date"].dt.day_name()
df_fe["Is_Weekend"] = df_fe["Day_of_Week"].isin(["Saturday", "Sunday"])
weekend_count = df_fe["Is_Weekend"].sum()
print(f"\n 8. Is Weekend:")
print(f"   TRUE (Weekend) : {weekend_count:,} ({weekend_count/len(df_fe)*100:.1f}%)")
print(f"   FALSE (Weekday): {len(df_fe)-weekend_count:,} ({(len(df_fe)-weekend_count)/len(df_fe)*100:.1f}%)")

# ============================================================
# 14. FEATURE 9: STAND
# ============================================================

stand_count = df_fe['Stand'].nunique()
print(f"\n 9. Stand: {stand_count} stand unik")
if stand_count > 1:
    print(df_fe["Stand"].value_counts().head(10))
else:
    print(f"   (Semua data memiliki Stand = {df_fe['Stand'].iloc[0] if len(df_fe) > 0 else 'UNKNOWN'})")

# ============================================================
# 15. FEATURE 10: MONTH
# ============================================================

df_fe["Month"] = df_fe["Departure_Date"].dt.month
df_fe["Month_Name"] = df_fe["Departure_Date"].dt.month_name()
print(f"\n 10. Month: {df_fe['Month'].nunique()} bulan")
print(df_fe["Month_Name"].value_counts())

# ============================================================
# 16. FEATURE 11: DAY OF WEEK
# ============================================================

df_fe["Day_of_Week_Num"] = df_fe["Departure_Date"].dt.dayofweek
print(f"\n 11. Day of Week:")
print(f"   Numerik : {df_fe['Day_of_Week_Num'].nunique()} nilai")
print(f"   Nama    : {df_fe['Day_of_Week'].nunique()} hari")
print(df_fe["Day_of_Week"].value_counts())

# ============================================================
# 17. FEATURE 12: DEPARTURE DELAY
# ============================================================

df_fe["Departure_Delay"] = df_fe.apply(
    lambda row: calc_delay(row["TakeOff_Time"], row["STD"]), axis=1
)

valid_count = df_fe['Departure_Delay'].notna().sum()
print(f"\n 12. Departure Delay: {valid_count:,} data valid")
if valid_count > 0:
    print(f"   Min  : {df_fe['Departure_Delay'].min():.0f} menit")
    print(f"   Max  : {df_fe['Departure_Delay'].max():.0f} menit")
    print(f"   Mean : {df_fe['Departure_Delay'].mean():.0f} menit")

# ============================================================
# 18. FEATURE 13: DELAY CATEGORY (Target - Aturan yang Benar)
# ============================================================

df_fe["Delay_Category"] = df_fe["Departure_Delay"].apply(categorize_delay_correct)

print(f"\n 13. Delay Category (Target - Aturan yang Benar):")
print(df_fe["Delay_Category"].value_counts().sort_index())

# ============================================================
# 19. CREATE DATASET FINAL
# ============================================================

print("\n" + "="*80)
print(" MEMBUAT DATASET FINAL")
print("="*80)

# Kolom yang akan diambil
columns_final = [
    'Arrival_Date', 'Arrival_Flight', 'Departure_Flight', 'Departure_REG',
    'Airline', 'Aircraft_Type', 'Origin', 'Destination',
    'Arrival_Delay', 'Turnaround_Time', 'Departure_Hour',
    'Departure_Period', 'Peak_Hour', 'Is_Weekend', 'Stand', 'Month',
    'Departure_Delay', 'Day_of_Week', 'Delay_Category'
]

# Filter kolom yang tersedia
columns_final_existing = [col for col in columns_final if col in df_fe.columns]
print(f"\n Kolom yang akan diambil ({len(columns_final_existing)} kolom):")
print(columns_final_existing)

# Buat dataset final
df_final = df_fe[columns_final_existing].copy()

print(f"\n Dataset Final:")
print(f"   Total baris : {len(df_final):,}")
print(f"   Total kolom : {len(df_final.columns)}")

# ============================================================
# 20. ENCODING
# ============================================================

from sklearn.preprocessing import LabelEncoder

# Delay Category Encoded
le_target = LabelEncoder()
df_final["Delay_Category_Encoded"] = le_target.fit_transform(df_final["Delay_Category"])
print(f"\n Delay Category Encoded:")
for i, cat in enumerate(le_target.classes_):
    print(f"   {i} : {cat}")

# Airline Encoded
le_airline = LabelEncoder()
df_final["Airline_Encoded"] = le_airline.fit_transform(df_final["Airline"])
print(f"\n Airline Encoded: {len(le_airline.classes_)} maskapai")

# ============================================================
# 21. STATISTIK AKHIR
# ============================================================

print("\n" + "="*80)
print(" STATISTIK AKHIR FEATURE ENGINEERING")
print("="*80)

print(f"\n Dataset Final (df_final):")
print(f"   Total baris : {len(df_final):,}")
print(f"   Total kolom : {len(df_final.columns)}")

print(f"\n Missing Value per Kolom:")
for col in df_final.columns:
    missing = df_final[col].isna().sum()
    if missing > 0:
        print(f"   {col:25s} : {missing:>5,} ({missing/len(df_final)*100:.1f}%)")
    else:
        print(f"   {col:25s} : {missing:>5,} ")




TAHAP 10: FEATURE ENGINEERING - FINAL VERSION (DIPERBAIKI)

 Data untuk Feature Engineering:
   Total baris : 2,574
   Total kolom : 30

----------------------------------------
 MENAMBAHKAN KOLOM STAND
----------------------------------------
Kolom Stand tidak ditemukan di df_fe, mengambil dari data mentah...

Kolom yang teridentifikasi di df_dep_clean:
   Flight Number : Flight_Number
   Data Date     : Data_Date
   Stand         : Stand

Mengambil Stand dari df_dep_clean

 Stand berhasil ditambahkan dari df_dep_clean
   Total Stand unik: 17

 Hasil kolom Stand:
   Total baris: 2,574
   Stand unik : 17
Stand
7      632
6      631
5      599
8      383
4      175
10      64
HGR     54
11       9
13       7
2        6
Name: count, dtype: int64

 CEK FORMAT KOLOM WAKTU:
   STA             dtype : object
   Landing_Time    dtype : object
   STD             dtype : object
   BlockOff_Time   dtype : object
   TakeOff_Time    dtype : object

 1. Airline: 26 maskapai
Airline
IU    791
QG   

In [47]:
# ============================================================
# 22. UPLOAD KE GOOGLE SHEETS
# ============================================================

print("\n" + "="*80)
print("UPLOAD KE GOOGLE SHEETS")
print("="*80)

# Buat salinan untuk upload
df_upload = df_final.copy()

# Konversi SEMUA kolom ke string
print("\n📌 Mengkonversi semua kolom ke string...")

for col in df_upload.columns:
    df_upload[col] = df_upload[col].astype(str)
    df_upload[col] = df_upload[col].str.replace('nan', '', case=False)
    df_upload[col] = df_upload[col].str.replace('None', '', case=False)
    df_upload[col] = df_upload[col].str.replace('NaT', '', case=False)
    df_upload[col] = df_upload[col].str.strip()

print("Semua kolom sudah dikonversi ke string")

# Upload ke Google Sheets
print("\n Mengupload ke Google Sheets...")

try:
    worksheet_final = sheet.worksheet("DATA_FINAL")
    worksheet_final.clear()
    print(" Worksheet DATA_FINAL ditemukan, akan di-update")
except:
    worksheet_final = sheet.add_worksheet(title="DATA_FINAL", rows="10000", cols="50")
    print(" Worksheet DATA_FINAL baru dibuat")

# Upload header
headers = df_upload.columns.tolist()
worksheet_final.append_row(headers)
print(f"   Header ({len(headers)} kolom) berhasil diupload")

# Upload data dalam batch
batch_size = 1000
total_rows = len(df_upload)
total_batches = (total_rows // batch_size) + 1

print(f"\n Mengupload {total_rows:,} baris data dalam {total_batches} batch...")

for i in range(0, total_rows, batch_size):
    batch = df_upload.iloc[i:i+batch_size].values.tolist()
    worksheet_final.append_rows(batch)
    print(f"   Batch {i//batch_size + 1}/{total_batches}: {len(batch)} baris")

print(f"\n Data berhasil diupload: {total_rows:,} baris, {len(headers)} kolom")
print(f"   Lokasi: Google Sheets → DATA_FINAL")


📤 UPLOAD KE GOOGLE SHEETS

📌 Mengkonversi semua kolom ke string...
✅ Semua kolom sudah dikonversi ke string

📤 Mengupload ke Google Sheets...
✅ Worksheet DATA_FINAL ditemukan, akan di-update
   Header (21 kolom) berhasil diupload

📊 Mengupload 2,574 baris data dalam 3 batch...
   Batch 1/3: 1000 baris
   Batch 2/3: 1000 baris
   Batch 3/3: 574 baris

✅ Data berhasil diupload: 2,574 baris, 21 kolom
   Lokasi: Google Sheets → DATA_FINAL


In [51]:
# ============================================================
# VISUALISASI DISTRIBUSI FEATURE ENGINEERING
# ============================================================
# Kode ini digunakan untuk melihat distribusi setiap fitur
# yang telah dibangun pada tahap Feature Engineering
# ============================================================

print("\n" + "="*80)
print("VISUALISASI DISTRIBUSI FEATURE ENGINEERING")
print("="*80)

# ============================================================
# 1. DISTRIBUSI AIRLINE
# ============================================================

print("\n" + "="*80)
print("1. DISTRIBUSI AIRLINE")
print("="*80)

print("\n Airline:")
print(df_fe["Airline"].value_counts())
print(f"\nTotal maskapai: {df_fe['Airline'].nunique()}")

# ============================================================
# 2. DISTRIBUSI AIRCRAFT TYPE
# ============================================================

print("\n" + "="*80)
print("2. DISTRIBUSI AIRCRAFT TYPE")
print("="*80)

print("\n Aircraft Type:")
print(df_fe["Aircraft_Type"].value_counts())
print(f"\nTotal tipe pesawat: {df_fe['Aircraft_Type'].nunique()}")

# ============================================================
# 3. DISTRIBUSI ARRIVAL DELAY
# ============================================================

print("\n" + "="*80)
print("3. DISTRIBUSI ARRIVAL DELAY")
print("="*80)

print("\nArrival Delay (menit):")
print(f"   Count  : {df_fe['Arrival_Delay'].count():,}")
print(f"   Mean   : {df_fe['Arrival_Delay'].mean():.2f}")
print(f"   Std    : {df_fe['Arrival_Delay'].std():.2f}")
print(f"   Min    : {df_fe['Arrival_Delay'].min():.2f}")
print(f"   Max    : {df_fe['Arrival_Delay'].max():.2f}")

# Kategorisasi Arrival Delay
def categorize_arrival_delay(delay):
    if pd.isna(delay):
        return "Unknown"
    if delay <= -30:
        return "Early (>30m)"
    elif delay <= -15:
        return "Early (15-30m)"
    elif delay <= 0:
        return "Early (0-15m)"
    elif delay <= 15:
        return "On Time (0-15m)"
    elif delay <= 30:
        return "Late (15-30m)"
    elif delay <= 60:
        return "Late (30-60m)"
    else:
        return "Late (>60m)"

df_fe["Arrival_Delay_Category"] = df_fe["Arrival_Delay"].apply(categorize_arrival_delay)
print("\n Kategori Arrival Delay:")
print(df_fe["Arrival_Delay_Category"].value_counts())

# ============================================================
# 4. DISTRIBUSI TURNAROUND TIME
# ============================================================

print("\n" + "="*80)
print("4. DISTRIBUSI TURNAROUND TIME")
print("="*80)

print("\n Turnaround Time (menit):")
print(f"   Count  : {df_fe['Turnaround_Time'].count():,}")
print(f"   Mean   : {df_fe['Turnaround_Time'].mean():.2f}")
print(f"   Std    : {df_fe['Turnaround_Time'].std():.2f}")
print(f"   Min    : {df_fe['Turnaround_Time'].min():.2f}")
print(f"   Max    : {df_fe['Turnaround_Time'].max():.2f}")

# Kategorisasi Turnaround Time
def categorize_tat(tat):
    if pd.isna(tat):
        return "Unknown"
    if tat < 30:
        return "Invalid (<30m)"
    elif tat <= 120:
        return "Normal (30-120m)"
    elif tat <= 360:
        return "Medium (2-6h)"
    elif tat <= 720:
        return "Long (6-12h)"
    else:
        return "Very Long (>12h)"

df_fe["TAT_Category_Dist"] = df_fe["Turnaround_Time"].apply(categorize_tat)
print("\n Distribusi Kategori Turnaround Time:")
print(df_fe["TAT_Category_Dist"].value_counts())

# ============================================================
# 5. DISTRIBUSI DEPARTURE HOUR
# ============================================================

print("\n" + "="*80)
print("5. DISTRIBUSI DEPARTURE HOUR")
print("="*80)

print("\n Departure Hour:")
print(df_fe["Departure_Hour"].value_counts().sort_index())

# ============================================================
# 6. DISTRIBUSI DEPARTURE PERIOD
# ============================================================

print("\n" + "="*80)
print("6. DISTRIBUSI DEPARTURE PERIOD")
print("="*80)

print("\n Departure Period:")
print(df_fe["Departure_Period"].value_counts())

# ============================================================
# 7. DISTRIBUSI PEAK HOUR
# ============================================================

print("\n" + "="*80)
print("7. DISTRIBUSI PEAK HOUR")
print("="*80)

peak_count = df_fe["Peak_Hour"].sum()
peak_pct = peak_count / len(df_fe) * 100
print(f"\n Peak Hour:")
print(f"   TRUE (Peak)   : {peak_count:,} ({peak_pct:.1f}%)")
print(f"   FALSE (Off Peak): {len(df_fe)-peak_count:,} ({100-peak_pct:.1f}%)")

# ============================================================
# 8. DISTRIBUSI IS WEEKEND
# ============================================================

print("\n" + "="*80)
print("8. DISTRIBUSI IS WEEKEND")
print("="*80)

weekend_count = df_fe["Is_Weekend"].sum()
weekend_pct = weekend_count / len(df_fe) * 100
print(f"\n Is Weekend:")
print(f"   TRUE (Weekend) : {weekend_count:,} ({weekend_pct:.1f}%)")
print(f"   FALSE (Weekday): {len(df_fe)-weekend_count:,} ({100-weekend_pct:.1f}%)")

# ============================================================
# 9. DISTRIBUSI STAND
# ============================================================

print("\n" + "="*80)
print("9. DISTRIBUSI STAND")
print("="*80)

print(f"\n Stand:")
print(f"   Total stand unik: {df_fe['Stand'].nunique()}")
print("\n   Top 10 Stand:")
print(df_fe["Stand"].value_counts().head(10))

# ============================================================
# 10. DISTRIBUSI MONTH
# ============================================================

print("\n" + "="*80)
print("10. DISTRIBUSI MONTH")
print("="*80)

print("\n Month:")
print(df_fe["Month_Name"].value_counts().sort_index())

# ============================================================
# 11. DISTRIBUSI DAY OF WEEK
# ============================================================

print("\n" + "="*80)
print("11. DISTRIBUSI DAY OF WEEK")
print("="*80)

print("\n Day of Week:")
print(df_fe["Day_of_Week"].value_counts())

# ============================================================
# 12. DISTRIBUSI DEPARTURE DELAY
# ============================================================

print("\n" + "="*80)
print("12. DISTRIBUSI DEPARTURE DELAY")
print("="*80)

print("\n Departure Delay (menit):")
print(f"   Count  : {df_fe['Departure_Delay'].count():,}")
print(f"   Mean   : {df_fe['Departure_Delay'].mean():.2f}")
print(f"   Std    : {df_fe['Departure_Delay'].std():.2f}")
print(f"   Min    : {df_fe['Departure_Delay'].min():.2f}")
print(f"   Max    : {df_fe['Departure_Delay'].max():.2f}")

# ============================================================
# 13. DISTRIBUSI DELAY CATEGORY (TARGET)
# ============================================================

print("\n" + "="*80)
print("13. DISTRIBUSI DELAY CATEGORY (TARGET)")
print("="*80)

print("\n Delay Category (Target - Aturan yang Benar):")
print(df_fe["Delay_Category"].value_counts().sort_index())

# Detail persentase
print("\n Persentase Delay Category:")
for cat, count in df_fe["Delay_Category"].value_counts().sort_index().items():
    pct = count / len(df_fe) * 100
    print(f"   {cat:>12} : {count:>5,} ({pct:>5.1f}%)")

# ============================================================
# 14. RINGKASAN STATISTIK SEMUA FITUR NUMERIK
# ============================================================

print("\n" + "="*80)
print("14. RINGKASAN STATISTIK SEMUA FITUR NUMERIK")
print("="*80)

numeric_cols = ['Arrival_Delay', 'Turnaround_Time', 'Departure_Hour', 'Departure_Delay']
print("\n Statistik Deskriptif Fitur Numerik:")
print(df_fe[numeric_cols].describe().round(2))

# ============================================================
# 15. RINGKASAN STATISTIK SEMUA FITUR KATEGORIKAL
# ============================================================

print("\n" + "="*80)
print("15. RINGKASAN STATISTIK FITUR KATEGORIKAL")
print("="*80)

categorical_cols = ['Airline', 'Aircraft_Type', 'Origin', 'Destination', 
                    'Departure_Period', 'Peak_Hour', 'Is_Weekend', 
                    'Stand', 'Month', 'Day_of_Week', 'Delay_Category']

print("\n Jumlah Unique Value per Fitur Kategorikal:")
for col in categorical_cols:
    if col in df_fe.columns:
        unique_count = df_fe[col].nunique()
        print(f"   {col:20s} : {unique_count:>5,} unique values")

# ============================================================
# 16. TABEL RINGKASAN SEMUA FITUR
# ============================================================

print("\n" + "="*80)
print("16. TABEL RINGKASAN SEMUA FITUR")
print("="*80)

print("\n┌─────────────────────────────────────────────────────────────────────────────┐")
print("│                    RINGKASAN FITUR FEATURE ENGINEERING                       │")
print("├─────────────────────────────────────────────────────────────────────────────┤")

fitur_list = [
    ("Airline", df_fe['Airline'].nunique(), "Kategorikal"),
    ("Aircraft Type", df_fe['Aircraft_Type'].nunique(), "Kategorikal"),
    ("Origin", df_fe['Origin'].nunique(), "Kategorikal"),
    ("Destination", df_fe['Destination'].nunique(), "Kategorikal"),
    ("Arrival Delay", f"{df_fe['Arrival_Delay'].mean():.0f} menit", "Numerik"),
    ("Turnaround Time", f"{df_fe['Turnaround_Time'].mean():.0f} menit", "Numerik"),
    ("Departure Hour", f"{df_fe['Departure_Hour'].min():.0f} - {df_fe['Departure_Hour'].max():.0f}", "Numerik"),
    ("Departure Period", df_fe['Departure_Period'].nunique(), "Kategorikal"),
    ("Peak Hour", f"{df_fe['Peak_Hour'].sum():,} ({df_fe['Peak_Hour'].sum()/len(df_fe)*100:.1f}%)", "Boolean"),
    ("Is Weekend", f"{df_fe['Is_Weekend'].sum():,} ({df_fe['Is_Weekend'].sum()/len(df_fe)*100:.1f}%)", "Boolean"),
    ("Stand", df_fe['Stand'].nunique(), "Kategorikal"),
    ("Month", df_fe['Month'].nunique(), "Numerik"),
    ("Day of Week", df_fe['Day_of_Week'].nunique(), "Kategorikal"),
    ("Departure Delay", f"{df_fe['Departure_Delay'].mean():.0f} menit", "Numerik"),
    ("Delay Category", df_fe['Delay_Category'].nunique(), "Target"),
]

for name, value, dtype in fitur_list:
    print(f"│ {name:20s} : {str(value):30s} {dtype:>10s} │")

print("└─────────────────────────────────────────────────────────────────────────────┘")

# ============================================================
# 17. SIMPAN HASIL DISTRIBUSI KE FILE (OPSIONAL)
# ============================================================

print("\n" + "="*80)
print("17. SIMPAN HASIL DISTRIBUSI")
print("="*80)

# Buat dictionary untuk menyimpan semua distribusi
distribusi = {
    'Airline': df_fe['Airline'].value_counts(),
    'Aircraft_Type': df_fe['Aircraft_Type'].value_counts(),
    'Departure_Period': df_fe['Departure_Period'].value_counts(),
    'Peak_Hour': df_fe['Peak_Hour'].value_counts(),
    'Is_Weekend': df_fe['Is_Weekend'].value_counts(),
    'Month': df_fe['Month_Name'].value_counts(),
    'Day_of_Week': df_fe['Day_of_Week'].value_counts(),
    'Delay_Category': df_fe['Delay_Category'].value_counts(),
    'TAT_Category': df_fe['TAT_Category_Dist'].value_counts(),
}

print("\n Distribusi telah dihitung dan siap digunakan untuk visualisasi.")

# ============================================================
# 18. REKOMENDASI VISUALISASI
# ============================================================

print("\n" + "="*80)
print("18. REKOMENDASI VISUALISASI UNTUK BAB 4")
print("="*80)

print("""
REKOMENDASI VISUALISASI:

1. Airline              → Bar Chart (Top 10 maskapai)
2. Aircraft Type        → Bar Chart (Top 8 tipe pesawat)
3. Arrival Delay        → Histogram / Boxplot
4. Turnaround Time      → Histogram / Boxplot + Kategori TAT
5. Departure Hour       → Histogram (distribusi jam)
6. Departure Period     → Pie Chart / Bar Chart
7. Peak Hour            → Pie Chart (TRUE vs FALSE)
8. Is Weekend           → Pie Chart (TRUE vs FALSE)
9. Stand                → Bar Chart (Top 10 stand)
10. Month               → Bar Chart (per bulan)
11. Day of Week         → Bar Chart (per hari)
12. Departure Delay     → Histogram / Boxplot
13. Delay Category      → Bar Chart (Target variable)
""")

print("\n" + "="*80)
print("✅ VISUALISASI DISTRIBUSI SELESAI")
print("="*80)


VISUALISASI DISTRIBUSI FEATURE ENGINEERING

1. DISTRIBUSI AIRLINE

 Airline:
Airline
IU         791
QG         450
GA         338
IP         279
AK         242
JT         165
ID         142
TR          54
SI          41
IW          41
AU           5
OE           4
UNKNOWN      4
OD           3
EC           2
AE           2
KF           2
BF           1
SO           1
PF           1
GL           1
BK           1
TT           1
AJ           1
EE           1
JE           1
Name: count, dtype: int64

Total maskapai: 26

2. DISTRIBUSI AIRCRAFT TYPE

 Aircraft Type:
Aircraft_Type
A320      1907
738        314
B738       148
E190        49
C208        42
A72         37
A330        13
B739        12
739          7
B773         5
320          5
AT72         4
B733F        4
A333         3
E135         2
733          2
C130         2
L600         2
G450         2
B773ER       2
E650         1
H850XP       1
G150         1
B200         1
B350         1
G7500        1
737          1
B73MAX       